# CS-781 Phase 6 (Abstractive Summarization) — Architecture 3: Transformer-XL (64M, 14 layers, d512), old BPE with the underscore fix - Phase-1 BERTScore 0.8350 / 0.8404 MBR

**Task.** Legal document → abstractive summary. Scored with **BERTScore F1** (roberta-large), plus ROUGE-1 / ROUGE-L F1.
Submission = `prediction.json`, a list of `{"id": ..., "summary": "..."}`, at the root of a zip, one summary per test document.

**Data** (checked in cell 4): 7,030 train documents with summaries, 100 test documents, no dev split, so 400 train documents are held out as dev.
Documents: median ~3.1k words, p90 ~7.9k. Summaries: median ~640 words (~22 % of the document), mostly built from the
document's own phrases. BERTScore reads only the first 512 roberta tokens (~400 words) of each text, so the
**start** of the summary is what counts → the model is trained on and generates the first ~480 tokens (~390 words).

**Long documents: extract-then-abstract.** A small selector (sklearn gradient boosting on position / length / TF-IDF
centrality / cue-word features, trained on my train split only) predicts which sentences re-appear in the summary.
The best sentences that fit the context budget are fed in document order. Cell 4 compares it with head / tail / head+tail /
centrality / oracle: how much of the reference each strategy even shows the model.

**Model.** My Phase-1 Transformer-XL, causal like in Phase 1 (relative positions, distances clamped at 511 = the largest it saw): one sequence = [soft prompt] + selected context + `Summary:` + summary, loss only on the summary.
Generation: KV cache, batched, n-gram blocking / repetition penalty; the decoding setting is chosen per job on dev (BERTScore).

**Experiments** (Phase-6 technique list; one Codabench zip each):

| job | technique | note |
|---|---|---|
| `lorma` | LoRMA on ["q_net", "kv_net"] (s = 1) | **compulsory**; KD teacher candidate |
| `full` | full fine-tuning | reference (not in the Phase-6 list) + KD teacher candidate |
| `kd` | knowledge distillation (token-level KL to the best teacher) → 6-layer student | **compulsory** |
| `kd_small` | KD → 4-layer student | student-size comparison |
| `lora` | LoRA on all 7 projections | |
| `qlora` | QLoRA: frozen weights in 4-bit NF4 + LoRA | |
| `adalora` | AdaLoRA (`peft`) | |
| `adapter_h` | Houlsby adapters (after attention + after FFN) | |
| `adapter_p` | Pfeiffer adapters (after the FFN only) | |
| `prompt` | prompt tuning (20 soft tokens) | |
| `prefix` | prefix tuning (16 vectors in every layer) | |
| `lorma_alt` | LoRMA with plain head truncation (or the selector, if head wins in cell 4) | long-document ablation |
| `baselines` | no training: zero-shot Phase-1 model, extractive selector, lead | analysis only, not submitted |


In [1]:
# ============================================================ 1. SETUP
import os, sys, json, time, shutil, subprocess, threading, pickle
from pathlib import Path
T_START = time.time()

QUICK_TEST = False        # True: tiny end-to-end check (~20-25 min).  False: the real run (commit it).
GROUP = "ALL"            # "ALL": all 13 jobs here (~10.5 h) | "A" + "B": the same jobs on two accounts (~10 h each, longer training)
SESSION_H = 12.0         # Kaggle session limit; jobs are shortened automatically so the notebook always finishes
ARCH, NORM = "txl", "ascii"
PHASE1_HINT = "a3-v2-oldjson"   # substring of the attached Phase-1 notebook's input path

WORK = Path("/kaggle/working/p6"); WORK.mkdir(parents=True, exist_ok=True)
CACHE = Path("/tmp/p6_cache") / "cache.pkl"; CACHE.parent.mkdir(parents=True, exist_ok=True)   # holds text -> NOT in the output
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
OUT = WORK / ("runs_quick" if QUICK_TEST else "runs")

def find(name, must_contain="", anywhere=False):
    hits = sorted(p for p in Path("/kaggle/input").rglob(name) if (anywhere or PHASE1_HINT in str(p).lower()) and must_contain in str(p))
    return str(hits[0]) if hits else None

PHASE1_CKPT = find("final.pt", "txl_full") or find("best.pt", "txl_full") or find("last.pt", "txl_full")
MODEL_PY    = find("txl_model.py")
if find("a1_tokenization.py"):                     # new BPE: a1_tokenization.py + tok/vocab.json + tok/merges.txt
    TOK_PY = find("a1_tokenization.py"); v = find("vocab.json"); TOK_PATH = str(Path(v).parent) if v else None
else:                                              # old BPE: the PATCHED a1_tokenizer.py + tok/bpe_tokenizer.json
    TOK_PY = find("a1_tokenizer.py"); TOK_PATH = find("bpe_tokenizer.json")
print("checkpoint :", PHASE1_CKPT); print("model code :", MODEL_PY); print("tokenizer  :", TOK_PY, "|", TOK_PATH)
assert PHASE1_CKPT and MODEL_PY, f"attach the Phase-1 notebook whose input path contains '{PHASE1_HINT}' (Add Input -> Your Work -> Notebooks), or fix PHASE1_HINT"
assert TOK_PY and TOK_PATH, "tokenizer files not found - attach the tokenizer dataset"

# Phase-6 data: a local copy if one is attached, otherwise download it with my HF secret (to the HF cache, outside /kaggle/working)
hits = sorted(Path("/kaggle/input").rglob("Phase_6_Summ/Train/train.json"))
if hits:
    DATA = hits[0].parent.parent; print("using local copy:", DATA)
else:
    HF_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        for k in ["HF", "HF_TOKEN"]:
            try: HF_TOKEN = UserSecretsClient().get_secret(k); break
            except Exception: pass
    except ImportError:
        HF_TOKEN = os.environ.get("HF_TOKEN")
    assert HF_TOKEN, "add your Hugging Face token as a Kaggle secret named HF (Add-ons -> Secrets) and tick it for this notebook"
    from huggingface_hub import snapshot_download
    DATA = Path(snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                                  allow_patterns=["Phase_6_Summ/*"])) / "Phase_6_Summ"
print("data:", DATA, sorted(str(p.relative_to(DATA)) for p in DATA.rglob("*.json")))
for pkg, imp in [("peft", "peft"), ("bert-score", "bert_score")]:
    try: __import__(imp)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
try:    # BERTScore model, downloaded once here so the parallel jobs only read the cache (used ONLY to score our dev split)
    from transformers import AutoModel, AutoTokenizer
    AutoTokenizer.from_pretrained("roberta-large"); _m = AutoModel.from_pretrained("roberta-large"); del _m
    print("roberta-large ready for the dev BERTScore")
except Exception as e:
    print("!!! roberta-large not available (", repr(e)[:120], ") -> dev selection falls back to ROUGE")


checkpoint : /kaggle/input/notebooks/alex595786/a3-v2-oldjson/txl_full/run/final.pt
model code : /kaggle/input/notebooks/alex595786/a3-v2-oldjson/txl/txl_model.py
tokenizer  : /kaggle/input/notebooks/alex595786/a3-v2-oldjson/txl/a1_tokenizer.py | /kaggle/input/notebooks/alex595786/a3-v2-oldjson/txl/tok/bpe_tokenizer.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

data: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ ['Test/test.json', 'Train/train.json']


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-large ready for the dev BERTScore


In [2]:
%%writefile /kaggle/working/p6/p2_ner.py
"""CS-781 Phase 2 (L-NER) on top of my Phase-1 models: Transformer-XL (arch=txl), decoder-only LegalLM (arch=dec),
the encoder of the encoder-decoder (arch=encdec), or RetNet (arch=ret).

One file, used two ways:
  * imported by the notebook  -> data prep, alignment checks, model-equivalence test
  * run as a script            -> `python p2_ner.py job.json` trains/evaluates/predicts ONE experiment

Pipeline
  text --(ASCII-fold + whitespace-collapse, exactly like Phase-1 pretraining, but keeping a char map)-->
  BPE tokens with ORIGINAL character offsets --> BIO tags (25) --> 512-token sliding windows -->
  Transformer-XL encoder (+ PEFT) + token head --> overlap-weighted log-prob merge per document -->
  BIO-constrained Viterbi --> character spans (whitespace-stripped) --> prediction.json

Adaptation methods (METHOD):
  full     full fine-tuning
  partial  top-N blocks + head (layer-wise / partial fine-tuning)
  lora     LoRA:  y = W x + s*B(A x)
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025, multiplicative):  y = (I + s*B A) W x   (B = 0 -> identity at init)
  adapter  Houlsby-style bottleneck adapters after attention and FFN sub-layers
  bitfit   biases (+ norms; TXL u/v biases). Bias-free LegalLM: zero-init biases added to every projection
  kd       knowledge distillation: a fine-tuned 14-layer teacher -> a shallower student built from Phase-1 layers
"""
import json
import math
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# ============================================================== text normalisation with char map
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")
# Phase-1 pre-tokenizer WITH the underscore fix that the Phase-1 notebook applied
PRE_RE = re.compile(r" ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+")


def normalize_with_map(text):
    """ASCII mode (Transformer-XL runs): same output as the TXL Phase-1 `normalize_text` (ASCII fold,
    straight quotes, single spaces, no newlines), plus omap[i] = ORIGINAL index of normalised char i."""
    norm, st, _ = normalize_ascii_map(text)
    return norm, st


def normalize_ascii_map(text):
    out, ost, oen = [], [], []
    pend = -1                                   # orig index of the first whitespace in a pending run
    for i, ch in enumerate(text):
        s = ch
        if not ch.isascii():
            s = ch.translate(_MAP)
            if not s.isascii():
                s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
        if s and _CTRL.search(s):
            s = _CTRL.sub("", s)
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


def normalize_nfkc_map(text, delete_table):
    """NFKC mode (decoder-only run): same output as a1_tokenization.normalize_text
    (delete Cc/Cf/Cs -> NFKC -> collapse whitespace -> strip). NFKC is applied per *cluster*
    (a char + its combining marks) so compositions match the whole-string NFKC.
    ost/oen[i] = ORIGINAL [start, end) of the cluster that produced normalised char i."""
    out, ost, oen = [], [], []
    pend = -1
    n, i = len(text), 0
    while i < n:
        j = i + 1
        while j < n and unicodedata.combining(text[j]):
            j += 1
        s = unicodedata.normalize("NFKC", text[i:j].translate(delete_table))
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(j)
        i = j
    return "".join(out), ost, oen


def align_map(orig, norm):
    """Character map norm -> orig from a diff (used only when the fast cluster map disagrees)."""
    import difflib
    st, en = [0] * len(norm), [0] * len(norm)
    sm = difflib.SequenceMatcher(None, orig, norm, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        for j in range(j1, j2):
            if tag == "equal":
                st[j], en[j] = i1 + (j - j1), i1 + (j - j1) + 1
            else:                                  # replaced / inserted: map to the whole changed region
                st[j], en[j] = i1, max(i2, i1 + 1) if i2 > i1 else min(i1 + 1, len(orig))
    return st, en


# RetNet Phase-1 preprocessing: ret_data.clean_text = remove "\r", collapse 3+ newlines to 2, strip (unicode kept)
_NL3 = re.compile(r"\n{3,}")
# the ORIGINAL a1_tokenizer pre-tokenizer (no underscore fix: "_" is never matched, i.e. dropped) - used by the RetNet run
PRE_RE_ORIG = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+")


def clean_text_ret(t):
    return _NL3.sub("\n\n", t.replace("\r", "")).strip()


def normalize_clean_map(text):
    idx = [i for i, c in enumerate(text) if c != "\r"]
    keep, run = [], 0
    for i in idx:                                              # 3+ consecutive newlines -> keep the first two
        if text[i] == "\n":
            run += 1
            if run > 2:
                continue
        else:
            run = 0
        keep.append(i)
    a, b = 0, len(keep)
    while a < b and text[keep[a]].isspace():
        a += 1
    while b > a and text[keep[b - 1]].isspace():
        b -= 1
    keep = keep[a:b]
    return "".join(text[i] for i in keep), keep, [i + 1 for i in keep]


# Archit-2-v2 preprocessing: " ".join(text.split())  (whitespace runs -> one space, ends stripped, unicode kept)
def normalize_ws_map(text):
    out, ost, oen = [], [], []
    pend = -1
    for i, ch in enumerate(text):
        if ch.isspace():
            if pend < 0:
                pend = i
        else:
            if pend >= 0 and out:
                out.append(" "); ost.append(pend); oen.append(pend + 1)
            pend = -1
            out.append(ch); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


class HFBPE:
    """Archit-2-v2 tokenizer: HuggingFace `tokenizers` ByteLevelBPETokenizer(vocab.json, merges.txt)
    (<pad>0 <s>1 </s>2 <unk>3), text = " ".join(text.split()). Offsets come from the tokenizer itself."""

    def __init__(self, tok_path, norm="ws"):
        from tokenizers import ByteLevelBPETokenizer
        p = Path(tok_path)
        p = p.parent if p.is_file() else p
        self.tok = ByteLevelBPETokenizer(str(p / "vocab.json"), str(p / "merges.txt"))
        self.kind, self.norm = "hf", norm
        self.vocab_size = self.tok.get_vocab_size()
        assert [self.tok.token_to_id(t) for t in ("<pad>", "<s>", "</s>", "<unk>")] == [0, 1, 2, 3]

    def normalize(self, text):
        n, st, en = normalize_ws_map(text)
        assert n == " ".join(text.split())
        return n, st, en

    def reference_encode(self, raw_text):
        return self.tok.encode(" ".join(raw_text.split())).ids


# ============================================================== tokenizer with offsets
class BPE:
    """Uniform wrapper around my two from-scratch Phase-1 tokenizers.
      kind='old' : a1_tokenizer.py  + bpe_tokenizer.json   (<pad>0 <unk>1 <s>2 </s>3; underscore-fixed regex)
      kind='new' : a1_tokenization.py + vocab.json/merges.txt (<pad>0 <s>1 </s>2 <unk>3; GPT-2 style regex)
    norm='ascii' : TXL pretraining preprocessing (ASCII fold)   |   norm='nfkc' : a1_tokenization.normalize_text
    norm='clean' : RetNet preprocessing (\r removal, newline collapse; ORIGINAL old regex, underscores dropped)
    Exposes: pre_re, encode_piece(piece) -> ids, nbytes[id], vocab_size, normalize(text), reference_encode(text)."""

    def __init__(self, tokenizer_py, tok_path, norm="ascii"):
        import importlib.util
        name = Path(tokenizer_py).stem
        spec = importlib.util.spec_from_file_location(name + "_p2", tokenizer_py)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        self.mod, self.norm = mod, norm
        if hasattr(mod, "BPETokenizer"):                              # old tokenizer
            self.kind = "old"
            p = Path(tok_path)
            self.tok = mod.BPETokenizer.load(str(p if p.is_file() else p / "bpe_tokenizer.json"))
            self.pre_re = PRE_RE_ORIG if norm == "clean" else PRE_RE      # RetNet run used the unpatched regex
            self.encode_piece = self.tok._encode_pretoken
            self.nbytes = [len(b) for b in self.tok.id_to_bytes]
            self.vocab_size = self.tok.get_vocab_size()
        else:                                                         # new tokenizer
            self.kind = "new"
            p = Path(tok_path)
            self.tok = mod.Tokenizer.load(str(p.parent if p.is_file() else p))
            self.pre_re = mod.PRETOKEN_RE
            self.encode_piece = self.tok._encode_word
            k = mod.N_SPECIAL
            self.nbytes = [0] * k + [len(t) for t in self.tok.id_to_token[k:]]   # 1 char = 1 byte (GPT-2 byte map)
            self.vocab_size = self.tok.vocab_size
        assert (norm == "ascii") or (norm == "nfkc" and self.kind == "new") or (norm == "clean" and self.kind == "old")

    def normalize(self, text):
        if self.norm == "nfkc":
            n, st, en = normalize_nfkc_map(text, self.mod._DELETE_TABLE)
            ref = self.mod.normalize_text(text)
            if n != ref:                       # rare: NFKC interaction across clusters -> exact diff alignment
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                st, en = align_map(text, ref)
                n = ref
            return n, st, en
        if self.norm == "clean":
            n, st, en = normalize_clean_map(text)
            if n != clean_text_ret(text):
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                n = clean_text_ret(text)
                st, en = align_map(text, n)
            return n, st, en
        return normalize_ascii_map(text)

    def encode_full(self, norm_text):
        out = []
        for piece in self.pre_re.findall(norm_text):
            out.extend(self.encode_piece(piece))
        return out

    def reference_encode(self, raw_text):
        """Exactly what Phase-1 pretraining fed the model for this raw text (used only for checks)."""
        if self.norm == "nfkc":
            return self.tok.encode(raw_text)                         # normalize_text + encode
        if self.norm == "clean":                                     # ret_data.clean_text + original a1_tokenizer.encode
            t = clean_text_ret(raw_text)
            return [i for piece in PRE_RE_ORIG.findall(t) for i in self.tok._encode_pretoken(piece)]
        return self.encode_full(self.normalize(raw_text)[0])


def load_bpe(tokenizer_py, tok_path, norm="ascii"):
    if norm == "ws":
        return HFBPE(tok_path, norm)
    return BPE(tokenizer_py, tok_path, norm)


def encode_doc(tok, text):
    """-> dict(ids int32[n], cs int32[n], ce int32[n]).  For token t, the ORIGINAL-text span is [cs, ce),
    where cs skips the single leading space a BPE token may carry. Handles tokens that split a
    multi-byte UTF-8 character (byte-level BPE) by mapping byte offsets to characters."""
    norm, ost, oen = tok.normalize(text)
    ids, cs, ce = [], [], []
    if getattr(tok, "kind", "") == "hf":                         # offsets straight from the HF tokenizer
        if not norm:
            return dict(ids=np.zeros(0, np.int32), cs=np.zeros(0, np.int32), ce=np.zeros(0, np.int32))
        enc = tok.tok.encode(norm)
        for tid, (a, z) in zip(enc.ids, enc.offsets):
            z = max(z, a + 1) - 1                                # last normalised char touched by this token
            a = min(a, len(norm) - 1); z = min(max(z, a), len(norm) - 1)
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid); cs.append(ost[a2]); ce.append(oen[z])
        return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))
    for m in tok.pre_re.finditer(norm):
        piece, p0 = m.group(), m.start()
        if piece.isascii():
            b2c = None
            nb = len(piece)
        else:
            b2c = [ci for ci, ch in enumerate(piece) for _ in range(len(ch.encode("utf-8")))]
            nb = len(b2c)
        bp = 0
        for tid in tok.encode_piece(piece):
            L = tok.nbytes[tid]
            a = p0 + (bp if b2c is None else b2c[bp])
            z = p0 + ((bp + L - 1) if b2c is None else b2c[bp + L - 1])     # last char touched by this token
            bp += L
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid)
            cs.append(ost[a2])
            ce.append(oen[z])
        assert bp == nb, "token byte lengths do not add up"
    return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))


# ============================================================== labels / BIO
def make_tags(labels):
    tags = ["O"] + [f"B-{l}" for l in labels] + [f"I-{l}" for l in labels]
    return tags, {t: i for i, t in enumerate(tags)}


def strip_span(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def bio_from_gold(text, enc, ents, tag2id):
    """Token BIO ids from character-level gold entities (entities are whitespace-stripped first)."""
    owner = np.full(len(text) + 1, -1, np.int32)
    lab_of = []
    for k, e in enumerate(ents):
        s, en = strip_span(text, int(e[0]), int(e[1]))
        owner[s:en] = k
        lab_of.append(e[2])
    cs = enc["cs"]
    y = np.zeros(len(cs), np.int64)
    prev = -1
    for t in range(len(cs)):
        k = owner[cs[t]] if cs[t] < len(text) else -1
        if k >= 0 and text[cs[t]].isspace() and k != prev:     # a whitespace-only token continues an entity, never starts one
            k = -1
        if k >= 0:
            y[t] = tag2id[("I-" if k == prev else "B-") + lab_of[k]]
        prev = k
    return y


def spans_from_tags(text, enc, tag_ids, tags):
    out, cur = [], None                     # cur = [first_tok, last_tok, label]
    for t, ti in enumerate(tag_ids):
        tg = tags[ti]
        if tg == "O":
            if cur: out.append(cur); cur = None
            continue
        bi, lab = tg.split("-", 1)
        if bi == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [t, t, lab]
        else:
            cur[1] = t
    if cur: out.append(cur)
    res = []
    for a, b, lab in out:
        s, e = strip_span(text, int(enc["cs"][a]), int(enc["ce"][b]))
        if e > s:
            res.append([s, e, lab])
    return res


def bio_transitions(tags):
    """allowed[i, j] for tag i -> tag j; start_ok[j]."""
    T = len(tags)
    allowed = np.ones((T, T), bool)
    start_ok = np.ones(T, bool)
    for j, tj in enumerate(tags):
        if tj.startswith("I-"):
            lab = tj[2:]
            start_ok[j] = False
            for i, ti in enumerate(tags):
                allowed[i, j] = ti in (f"B-{lab}", f"I-{lab}")
    return allowed, start_ok


def viterbi(logp, allowed, start_ok):
    """logp [n, T] log-probs -> best BIO-valid tag sequence."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, np.int64)
    trans = np.where(allowed, 0.0, -1e9)
    score = np.where(start_ok, logp[0], -1e9)
    back = np.zeros((n, T), np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans            # [from, to]
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    best = np.zeros(n, np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best


# ============================================================== metric (strict span + label)
def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = {(int(s), int(e), l) for s, e, l, *_ in gold[did]}
        p = {(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l])
    mac = lambda k: sum(per[l][k] for l in labels) / len(labels)
    return dict(macro_f1=mac("f1"), macro_p=mac("p"), macro_r=mac("r"), per_label=per)


def error_breakdown(gold, pred):
    """Classify every false positive: label error (same span, other label), boundary error
    (overlaps a gold span of the same label), or spurious; plus the label-confusion counts."""
    kinds, conf = Counter(), Counter()
    for did in gold:
        g = [(int(s), int(e), l) for s, e, l, *_ in gold[did]]
        gs = set(g)
        for s, e, l in [(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])]:
            if (s, e, l) in gs:
                continue
            same = [gl for (a, b, gl) in g if a == s and b == e]
            if same:
                kinds["label_error"] += 1; conf[f"{same[0]}->{l}"] += 1
            elif any(a < e and s < b and gl == l for a, b, gl in g):
                kinds["boundary_error"] += 1
            elif any(a < e and s < b for a, b, gl in g):
                kinds["overlap_other_label"] += 1
            else:
                kinds["spurious"] += 1
    return dict(fp_kinds=dict(kinds), top_confusions=conf.most_common(15))


# ============================================================== data prep (called from the notebook)
def prepare(data_dir, tok, labels, cache_path):
    data_dir = Path(data_dir)
    tags, tag2id = make_tags(labels)
    out = {}
    for fold in ["fold1", "fold2", "fold3"]:
        sub = "Test" if fold == "fold3" else "Train"
        texts = json.load(open(data_dir / sub / f"{fold}-text.json", encoding="utf-8"))
        gold = json.load(open(data_dir / sub / f"{fold}-gold.json", encoding="utf-8")) if fold != "fold3" else None
        docs = {}
        for did, text in texts.items():
            enc = encode_doc(tok, text)
            d = dict(text=text, **enc)
            if gold is not None:
                d["gold"] = gold.get(did, [])
                d["y"] = bio_from_gold(text, enc, d["gold"], tag2id)
            docs[did] = d
        out[fold] = docs
    pickle.dump(dict(labels=labels, tags=tags, folds=out), open(cache_path, "wb"))
    return out


def alignment_report(folds, labels):
    """Upper bound of strict macro-F1 if the model predicted the gold BIO tags perfectly."""
    tags, _ = make_tags(labels)
    rep = {}
    for fold in ["fold1", "fold2"]:
        gold = {d: v["gold"] for d, v in folds[fold].items()}
        pred = {d: spans_from_tags(v["text"], v, v["y"], tags) for d, v in folds[fold].items()}
        sc = strict_scores(gold, pred, labels)
        miss = []
        for d, v in folds[fold].items():
            p = {tuple(x) for x in pred[d]}
            for s, e, l, *_ in v["gold"]:
                if (s, e, l) not in p:
                    miss.append((d, s, e, l, v["text"][max(0, s - 15):e + 15]))
        rep[fold] = dict(upper_bound_macro_f1=sc["macro_f1"], n_gold=sum(len(v["gold"]) for v in folds[fold].values()),
                         n_unrecoverable=len(miss), examples=miss[:12])
    ntok = {f: int(sum(len(v["ids"]) for v in folds[f].values())) for f in folds}
    rep["tokens_per_fold"] = ntok
    return rep


# ============================================================== torch parts
import torch                                   # noqa: E402
import torch.nn as nn                          # noqa: E402
import torch.nn.functional as F                # noqa: E402


def _import(path, name):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def load_phase1(txl_model_py, ckpt_path):
    """Transformer-XL Phase-1 checkpoint: {'model': state_dict, 'config': dict}."""
    mod = _import(txl_model_py, "txl_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = mod.TXLConfig(**ck["config"])
    m = mod.TransformerXL(cfg)
    m.load_state_dict(ck["model"])
    return m, mod


def load_decoder(legal_lm_py, weights_path):
    """Decoder-only LegalLM Phase-1 weights (run1/final/model.pt + model_config.json)."""
    mod = _import(legal_lm_py, "legal_lm_p2")
    m = mod.load_legal_lm(weights_path, device="cpu", dtype=torch.float32)
    return m.train(), mod


def load_encdec(edmodel_py, ckpt_path):
    """Encoder-decoder Phase-1 checkpoint (phase1_encdec_v2/best.pt: {'model', 'cfg', ...})."""
    mod = _import(edmodel_py, "edmodel_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.EncDecLM(mod.EDConfig(**ck["cfg"]))
    m.load_state_dict(ck["model"])
    return m, mod


def load_retnet(ret_model_py, ckpt_path):
    """RetNet Phase-1 checkpoint (retnet/ckpt/best.pt: {'model', 'config': {'model': ...}, 'vocab_size', ...})."""
    mod = _import(ret_model_py, "ret_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.RetNet(ck["vocab_size"], **ck["config"]["model"])
    sd = ck["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    m.load_state_dict(sd)
    return m, mod


# ---- Archit-2-v2 encoder-decoder (same module/parameter names as the Phase-1 notebook, generation code removed)
class _A2Emb(nn.Module):
    def __init__(self, vocab_size, d_model, max_length, pad_id, dropout):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, ids, offset=0):
        T = ids.size(1)
        pos = torch.arange(offset, offset + T, device=ids.device).unsqueeze(0)
        return self.dropout(self.token_embedding(ids) * math.sqrt(self.d_model) + self.position_embedding(pos))


class _A2MHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        self.d_model, self.h, self.dh, self.p = d_model, num_heads, d_model // num_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.dh).transpose(1, 2)

    def forward(self, query, key_value=None, key_padding_mask=None, causal=False):
        kv = query if key_value is None else key_value
        q, k, v = self._split(self.q_proj(query)), self._split(self.k_proj(kv)), self._split(self.v_proj(kv))
        mask = None
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].bool()
            if causal:
                cm = torch.tril(torch.ones(query.size(1), kv.size(1), dtype=torch.bool, device=query.device))[None, None]
                mask = mask & cm
                causal = False
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.p if self.training else 0.0, is_causal=causal)
        B, _, T, _ = o.shape
        return self.out_proj(o.transpose(1, 2).contiguous().view(B, T, self.d_model))


class _A2FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))


class _A2EncLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.ffn, self.dropout2 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, src_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=src_mask))
        return x + self.dropout2(self.ffn(self.norm2(x)))


class _A2DecLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.cross_attn, self.dropout2 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm3, self.ffn, self.dropout3 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, enc_out, src_mask, self_mask=None):
        """self_mask=None -> causal (= Phase-1). Otherwise a key-padding mask -> bidirectional decoder."""
        if self_mask is None:
            x = x + self.dropout1(self.self_attn(self.norm1(x), causal=True))
        else:
            x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=self_mask))
        x = x + self.dropout2(self.cross_attn(self.norm2(x), key_value=enc_out, key_padding_mask=src_mask))
        return x + self.dropout3(self.ffn(self.norm3(x)))


class _A2Stack(nn.Module):
    def __init__(self, emb, layers, d):
        super().__init__()
        self.embedding, self.layers, self.final_norm = emb, nn.ModuleList(layers), nn.LayerNorm(d)


class A2V2EncDec(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, heads=12, enc_layers=6, dec_layers=8, d_ff=3072,
                 dropout=0.05, enc_max=1024, dec_max=256, pad=0):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, d_model=d_model, heads=heads, enc_layers=enc_layers,
                        dec_layers=dec_layers, d_ff=d_ff, enc_max=enc_max, dec_max=dec_max)
        ee = _A2Emb(vocab_size, d_model, enc_max, pad, dropout)
        de = _A2Emb(vocab_size, d_model, dec_max, pad, dropout)
        de.token_embedding.weight = ee.token_embedding.weight
        self.encoder = _A2Stack(ee, [_A2EncLayer(d_model, heads, d_ff, dropout) for _ in range(enc_layers)], d_model)
        self.decoder = _A2Stack(de, [_A2DecLayer(d_model, heads, d_ff, dropout) for _ in range(dec_layers)], d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = de.token_embedding.weight

    def encode(self, ids, src_mask):
        x = self.encoder.embedding(ids)
        for l in self.encoder.layers:
            x = l(x, src_mask)
        return self.encoder.final_norm(x)

    def decode_hidden(self, ids, enc, src_mask):
        x = self.decoder.embedding(ids)
        for l in self.decoder.layers:
            x = l(x, enc, src_mask)
        return self.decoder.final_norm(x)


def load_encdec2(ckpt_path):
    """Archit-2-v2 best_model.pt (a plain state_dict). Shapes are read from the weights (v2: 768/6/8/3072; v1: 512/6/6/2048)."""
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model" in sd and not any(k.startswith("encoder.") for k in sd):
        sd = sd["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    V, d = sd["encoder.embedding.token_embedding.weight"].shape
    ne = len({k.split(".")[2] for k in sd if k.startswith("encoder.layers.")})
    nd = len({k.split(".")[2] for k in sd if k.startswith("decoder.layers.")})
    m = A2V2EncDec(vocab_size=V, d_model=d, heads=d // 64, enc_layers=ne, dec_layers=nd,
                   d_ff=sd["encoder.layers.0.ffn.fc1.weight"].shape[0],
                   enc_max=sd["encoder.embedding.position_embedding.weight"].shape[0],
                   dec_max=sd["decoder.embedding.position_embedding.weight"].shape[0])
    m.load_state_dict(sd, strict=True)
    return m


def load_backbone(J):
    if J.get("arch", "txl") == "encdec2":
        return load_encdec2(J["phase1_ckpt"])
    if J.get("arch", "txl") == "ret":
        return load_retnet(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "encdec":
        return load_encdec(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "dec":
        return load_decoder(J["model_py"], J["phase1_ckpt"])[0]
    return load_phase1(J.get("model_py", J.get("txl_model_py")), J["phase1_ckpt"])[0]


class LowRankLinear(nn.Module):
    """mode='lora'  : y = W x + b + s * B(A x)
       mode='lorma' : y = (I + s * B A) (W x) + b       (LoRMA_+, multiplicative, identity at init)"""

    def __init__(self, base: nn.Linear, r, alpha, dropout, mode):
        super().__init__()
        self.base, self.mode, self.s = base, mode, alpha / r
        for p in base.parameters():
            p.requires_grad_(False)
        din = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, din))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == "lora":
            return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.s * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class Adapter(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a sub-layer projection."""

    def __init__(self, base: nn.Linear, bottleneck):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


class BiasLinear(nn.Module):
    """BitFit for bias-free projections: frozen W plus a trainable, zero-initialised bias."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return self.base(x) + self.bias


class TaggerHead(nn.Module):
    def _init_head(self, d, n_tags, dropout, head):
        self.drop = nn.Dropout(dropout)
        self.head_type = head
        if head == "bilstm":
            self.lstm = nn.LSTM(d, 256, batch_first=True, bidirectional=True)
            self.head = nn.Linear(512, n_tags)
        else:
            self.head = nn.Linear(d, n_tags)

    def forward(self, ids, attn_mask):
        h = self.hidden(ids, attn_mask)
        if self.head_type == "bilstm":
            h, _ = self.lstm(self.drop(h))
        return self.head(self.drop(h))


class TXLTagger(TaggerHead):
    """Transformer-XL (Phase-1 weights) used as a token encoder.
    attn='bidir' : every token sees left AND right context; relative distances are extended to negative
                   values (the same sinusoid / r_net / v-bias parametrisation, just evaluated at d<0).
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.pos_emb, self.ln_f = base.emb, base.pos_emb, base.ln_f
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.final_norm = self.ln_f
        self._init_head(self.d, n_tags, dropout, head)

    def _block(self, blk, h, r, gidx, mask):
        a = blk.attn
        B, L, _ = h.shape
        H, dh = a.H, a.dh
        x = blk.ln1(h)
        q = a.q_net(x).view(B, L, H, dh).transpose(1, 2)
        kv = a.kv_net(x).view(B, L, 2, H, dh)
        k, v = kv[:, :, 0].transpose(1, 2), kv[:, :, 1].transpose(1, 2)
        rk = a.r_net(r).view(-1, H, dh).transpose(0, 1).unsqueeze(0)          # [1,H,2L-1,dh]
        qu = (q + a.u[None, :, None, :]) * a.scale
        qv = (q + a.v[None, :, None, :]) * a.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                              # [B,H,L,L]
        BDf = torch.matmul(qv, rk.transpose(-1, -2))                            # [B,H,L,2L-1]
        BD = BDf.gather(-1, gidx.expand(B, H, L, L))
        score = (AC + BD).float().masked_fill(mask, -1e30)
        prob = a.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, L, H * dh)
        h = h + a.drop(a.o_net(out))
        h = h + blk.drop(blk.ff2(blk.drop(F.gelu(blk.ff1(blk.ln2(h))))))
        return h

    def hidden(self, ids, attn_mask):
        """ids [B,L] long, attn_mask [B,L] bool (True = real token) -> final hidden [B,L,D]."""
        B, L = ids.shape
        dev = ids.device
        h = self.emb(ids)
        pos = torch.arange(L - 1, -L, -1, device=dev, dtype=torch.float32)       # distance d = L-1-c
        r = self.pos_emb(pos)
        i = torch.arange(L, device=dev)
        gidx = ((L - 1) - i[:, None] + i[None, :])[None, None]                   # c for distance i-j
        mask = (~attn_mask)[:, None, None, :]                                    # [B,1,1,L]
        if self.attn_mode == "causal":
            mask = mask | torch.ones(L, L, dtype=torch.bool, device=dev).triu(1)[None, None]
        for blk in self.blocks:
            h = self._block(blk, h, r, gidx, mask)
        return self.ln_f(h)


class DecTagger(TaggerHead):
    """Decoder-only LegalLM (RoPE, RMSNorm, SwiGLU; Phase-1 weights) used as a token encoder.
    attn='bidir' : causal mask removed (RoPE is relative, so left and right distances are both defined)
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.cfg = base.cfg
        self.tok_emb, self.norm = base.tok_emb, base.norm
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        self.rope_mod = sys.modules[type(base).__module__]
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.norm
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        B, T = ids.shape
        dev = ids.device
        hd = self.cfg.d_model // self.cfg.n_head
        pos = torch.arange(T, device=dev)[None]
        cos, sin = self.rope_mod.rope_cos_sin(pos, hd, self.cfg.rope_theta)
        allow = attn_mask[:, None, None, :]                                   # [B,1,1,T] True = may attend
        if self.attn_mode == "causal":
            allow = allow & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        allow = allow | torch.eye(T, dtype=torch.bool, device=dev)[None, None]  # pad rows never fully masked
        x = self.tok_emb(ids)
        for blk in self.blocks:
            a = blk.attn
            hx = blk.attn_norm(x)
            q = a.q_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            k = a.k_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            v = a.v_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            q, k = self.rope_mod.apply_rope(q, cos, sin), self.rope_mod.apply_rope(k, cos, sin)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=allow)
            x = x + blk.drop(a.o_proj(y.transpose(1, 2).reshape(B, T, -1)))
            x = x + blk.drop(blk.mlp(blk.mlp_norm(x)))
        return self.norm(x)


class EncTagger(TaggerHead):
    """Phase-1 encoder-decoder as a tagger.
    use_decoder=False : encoder only (natively bidirectional).
    use_decoder=True  : BART-style. The window goes through the encoder AND, as decoder input, through the decoder
                        (self-attention + cross-attention to the encoder); tags are read from the DECODER states.
                        attn='bidir' removes the decoder's causal mask, attn='causal' keeps it (= Phase-1 computation).
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        self.cfg = base.c
        self.tok, self.enc_pos, self.enc_norm = base.tok, base.enc_pos, base.enc_norm
        ke = list(range(len(base.enc_layers))) if layers is None else [i for i in layers if i < len(base.enc_layers)]
        self.enc_blocks = nn.ModuleList([base.enc_layers[i] for i in ke])
        if use_decoder:
            self.dec_pos, self.dec_norm = base.dec_pos, base.dec_norm
            kd = list(range(len(base.dec_layers))) if layers is None else [i for i in layers if i < len(base.dec_layers)]
            self.dec_blocks = nn.ModuleList([base.dec_layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))   # for PEFT / partial
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = base.c.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.emb_drop = nn.Dropout(dropout)
        self._init_head(self.d, n_tags, dropout, head)

    def encode(self, ids, attn_mask):
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.enc_pos(torch.arange(T, device=dev))[None])
        mask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal" and not self.use_decoder:
            mask = mask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        mask = mask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.enc_blocks:
            x = l(x, mask)
        return self.enc_norm(x)

    def hidden(self, ids, attn_mask):
        mem = self.encode(ids, attn_mask)
        if not self.use_decoder:
            return mem
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.dec_pos(torch.arange(T, device=dev))[None])
        cmask = attn_mask[:, None, None, :]                                     # cross-attn: real encoder tokens
        smask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal":
            smask = smask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        smask = smask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.dec_blocks:
            x = x + l.drop(l.self_attn(l.n1(x), mask=smask))
            x = x + l.drop(l.cross(l.n2(x), mem=mem, mask=cmask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.dec_norm(x)


class ED2Tagger(TaggerHead):
    """Archit-2-v2 encoder-decoder as a tagger.
    use_decoder=False : tags from the ENCODER (natively bidirectional, 1024 positions). attn must be 'bidir'.
    use_decoder=True  : BART-style. Window -> encoder, same window -> decoder (cross-attends to the encoder);
                        tags from the decoder states. attn='causal' = Phase-1 computation. Decoder has 256 positions.
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        enc = base.encoder
        self.enc_emb, self.enc_norm = enc.embedding, enc.final_norm
        ke = list(range(len(enc.layers))) if layers is None else [i for i in layers if i < len(enc.layers)]
        self.enc_blocks = nn.ModuleList([enc.layers[i] for i in ke])
        if use_decoder:
            dec = base.decoder
            self.dec_emb, self.dec_norm = dec.embedding, dec.final_norm
            kd = list(range(len(dec.layers))) if layers is None else [i for i in layers if i < len(dec.layers)]
            self.dec_blocks = nn.ModuleList([dec.layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = enc.embedding.d_model
        for m in list(self.blocks.modules()) + [self.enc_emb.dropout] + ([self.dec_emb.dropout] if use_decoder else []):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        x = self.enc_emb(ids)
        for l in self.enc_blocks:
            x = l(x, attn_mask)
        mem = self.enc_norm(x)
        if not self.use_decoder:
            return mem
        y = self.dec_emb(ids)
        sm = None if self.attn_mode == "causal" else attn_mask
        for l in self.dec_blocks:
            y = l(y, mem, attn_mask, self_mask=sm)
        return self.dec_norm(y)


class RetTagger(TaggerHead):
    """RetNet (Phase-1 weights) as a token encoder, parallel form of multi-scale retention over a window.
    attn='causal': D_nm = gamma^(n-m) for n>=m  (exactly the Phase-1 computation; verified in the notebook)
    attn='bidir' : D_nm = gamma^|n-m|           (symmetric decay, so every token also retains its right context)
    Padding uses document id -1, so the same-document mask of RetNet also removes pads."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.final_ln = base.emb, base.final_ln
        self.register_buffer("inv_freq", base.inv_freq.clone(), persistent=False)
        keep = list(range(len(base.layers))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.layers[i] for i in keep])
        self.d = base.d_model
        self.rot = sys.modules[type(base).__module__].rotate
        self.emb_drop = nn.Dropout(dropout)
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.final_ln
        self._init_head(self.d, n_tags, dropout, head)

    def _msr(self, m, x, cos, sin, doc):
        B, L, _ = x.shape
        H, dk, dv = m.H, m.dk, m.dv
        q = m.wq(x).view(B, L, H, dk)
        k = m.wk(x).view(B, L, H, dk)
        v = m.wv(x).view(B, L, H, dv)
        g = m.wg(x)
        with torch.autocast(device_type=x.device.type, enabled=False):
            q = (self.rot(q.float(), cos, sin) * dk ** -0.5).transpose(1, 2)
            k = self.rot(k.float(), cos, sin).transpose(1, 2)
            v = v.float().transpose(1, 2)
            i = torch.arange(L, device=x.device, dtype=torch.float32)
            diff = i.view(L, 1) - i.view(1, L)
            lg = m.log_gamma.view(H, 1, 1)
            if self.attn_mode == "causal":
                D = torch.exp(lg * diff.clamp(min=0)) * (diff >= 0)
            else:
                D = torch.exp(lg * diff.abs())
            same = (doc.unsqueeze(2) == doc.unsqueeze(1)).unsqueeze(1)
            o = torch.matmul(torch.matmul(q, k.transpose(-1, -2)) * D.unsqueeze(0) * same, v)
            o = o.transpose(1, 2).reshape(B * L, H * dv)
            o = F.group_norm(o, H, m.gn_w.float(), m.gn_b.float(), 1e-5)
        o = o.view(B, L, H * dv)
        return m.wo(F.silu(g) * o.to(g.dtype))

    def hidden(self, ids, attn_mask):
        B, L = ids.shape
        doc = torch.where(attn_mask, torch.zeros_like(ids), torch.full_like(ids, -1))
        pos = torch.arange(L, device=ids.device)[None].expand(B, L)
        ang = pos.float().unsqueeze(-1) * self.inv_freq
        cos, sin = ang.cos().unsqueeze(2), ang.sin().unsqueeze(2)
        h = self.emb_drop(self.emb(ids))
        for blk in self.blocks:
            h = h + blk.drop(self._msr(blk.msr, blk.ln1(h), cos, sin, doc))
            y = blk.ln2(h)
            h = h + blk.drop(blk.down(F.silu(blk.gate(y)) * blk.up(y)))
        return self.final_ln(h)


def _linear_slots(blk, names):
    """(holder_module, attribute) for every nn.Linear attribute in `names` inside one block."""
    out = []
    for mod in blk.modules():
        for nm in names:
            if isinstance(getattr(mod, nm, None), nn.Linear) and not isinstance(mod, (LowRankLinear, Adapter, BiasLinear)):
                out.append((mod, nm))
    return out


def apply_method(model, method, J):
    """Freeze / wrap according to the adaptation method. The head is always trainable."""
    for p in model.parameters():
        p.requires_grad_(method in ("full", "kd"))
    if method == "partial":
        for blk in list(model.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in model.final_norm.parameters():
            p.requires_grad_(True)
    elif method in ("lora", "lorma"):
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                setattr(holder, nm, LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method))
    elif method == "adapter":
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J.get("adapter_targets", ["o_net", "ff2"])):
                setattr(holder, nm, Adapter(getattr(holder, nm), J["adapter_dim"]))
    elif method == "bitfit":
        if J.get("arch", "txl") in ("dec", "ret"):            # bias-free model: add zero-init biases + norms
            for blk in model.blocks:
                for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                    setattr(holder, nm, BiasLinear(getattr(holder, nm)))
        for n, p in model.named_parameters():
            if n.endswith("bias") or n.endswith(".u") or n.endswith(".v") or n.endswith(".gn_w") or n.endswith(".gn_b"):
                p.requires_grad_(True)
        for m in model.modules():                              # every LayerNorm / RMSNorm
            if "Norm" in type(m).__name__:
                for p in m.parameters():
                    p.requires_grad_(True)
    for p in model.head.parameters():
        p.requires_grad_(True)
    if getattr(model, "lstm", None) is not None:
        for p in model.lstm.parameters():
            p.requires_grad_(True)
    return model


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# ============================================================== windows / batching
def windows_for(n, L, stride):
    if n <= L:
        return [0]
    st = list(range(0, n - L + 1, stride))
    if st[-1] + L < n:
        st.append(n - L)
    return st


def make_train_windows(docs, L, stride):
    W = []
    for d in docs.values():
        n = len(d["ids"])
        for s in windows_for(n, L, stride):
            W.append((d["ids"][s:s + L], d["y"][s:s + L]))
    return W


def collate(items, with_y=True):
    L = max(len(x[0]) for x in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    am = torch.zeros(len(items), L, dtype=torch.bool)
    y = torch.full((len(items), L), -100, dtype=torch.long)
    for b, it in enumerate(items):
        n = len(it[0])
        ids[b, :n] = torch.as_tensor(it[0], dtype=torch.long)
        am[b, :n] = True
        if with_y:
            y[b, :n] = torch.as_tensor(it[1], dtype=torch.long)
    return ids, am, y


@torch.no_grad()
def doc_logprobs(model, docs, L, stride, bs, device, amp=True):
    """Merge per-window log-probs into one [n, T] array per doc (triangular weights: tokens near a window
    edge get little weight, so every token is mostly judged with context on both sides)."""
    model.eval()
    jobs = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n:
            for s in windows_for(n, L, stride):
                jobs.append((did, s, min(L, n - s)))
    T = model.head.out_features
    acc = {did: np.zeros((len(d["ids"]), T), np.float32) for did, d in docs.items()}
    wsum = {did: np.zeros((len(d["ids"]), 1), np.float32) for did, d in docs.items()}
    jobs.sort(key=lambda j: -j[2])
    for k in range(0, len(jobs), bs):
        chunk = jobs[k:k + bs]
        items = [(docs[did]["ids"][s:s + n], None) for did, s, n in chunk]
        ids, am, _ = collate(items, with_y=False)
        ids, am = ids.to(device), am.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            lg = model(ids, am)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for b, (did, s, n) in enumerate(chunk):
            pos = np.arange(n)
            w = np.minimum(pos + 1, n - pos).astype(np.float32)[:, None]
            if s == 0:
                w[: n // 2] = np.maximum(w[: n // 2], n / 2)        # doc start: no left context exists anyway
            if s + n == len(docs[did]["ids"]):
                w[n // 2:] = np.maximum(w[n // 2:], n / 2)          # doc end
            acc[did][s:s + n] += lp[b, :n] * w
            wsum[did][s:s + n] += w
    return {did: acc[did] / np.maximum(wsum[did], 1e-6) for did in docs}


def decode_docs(docs, lps, tags, use_viterbi=True):
    allowed, start_ok = bio_transitions(tags)
    pred = {}
    for did, d in docs.items():
        lp = lps[did]
        tg = viterbi(lp, allowed, start_ok) if use_viterbi else lp.argmax(-1)
        pred[did] = spans_from_tags(d["text"], d, tg, tags)
    return pred


# ============================================================== training
def class_weights(windows, T, mode):
    if mode == "none":
        return None
    cnt = np.bincount(np.concatenate([w[1] for w in windows]), minlength=T).astype(np.float64)
    w = np.ones(T)
    ent = cnt[1:]
    mean = ent[ent > 0].mean()
    w[1:] = np.clip(np.sqrt(mean / np.maximum(ent, 1)), 0.5, 4.0)
    return torch.tensor(w, dtype=torch.float32)


def build_model(J, base_factory, n_tags, method):
    base = base_factory()
    layers = J["kd_student_layers"] if method == "kd" else None
    cls = {"dec": DecTagger, "encdec": EncTagger, "encdec2": ED2Tagger, "ret": RetTagger}.get(J.get("arch", "txl"), TXLTagger)
    kw = dict(use_decoder=J.get("encdec_use_decoder", False)) if cls in (EncTagger, ED2Tagger) else {}
    if cls is ED2Tagger and not kw["use_decoder"]:
        assert J["attn"] == "bidir", "encoder-only tagging is always bidirectional"
    m = cls(base, n_tags, attn=J["attn"], layers=layers, dropout=J["dropout"], head=J["head"], **kw)
    return apply_method(m, method, J)


def train_one(J, model, train_docs, val_docs, tags, labels, device, epochs, log, teacher=None):
    torch.manual_seed(J["seed"]); np.random.seed(J["seed"])
    model.to(device)
    W = make_train_windows(train_docs, J["max_len"], J["train_stride"])
    cw = class_weights(W, len(tags), J["class_weight"])
    cw = cw.to(device) if cw is not None else None
    head_names = {n for n, _ in model.named_parameters() if n.startswith("head") or n.startswith("lstm")}
    new_ids = set()
    for mm in model.modules():
        if isinstance(mm, LowRankLinear):
            new_ids |= {id(mm.A), id(mm.B)}
        elif isinstance(mm, Adapter):
            new_ids |= {id(p) for p in list(mm.down.parameters()) + list(mm.up.parameters())}
    new_names = {n for n, p in model.named_parameters() if id(p) in new_ids}
    g_head = [p for n, p in model.named_parameters() if p.requires_grad and (n in head_names or n in new_names)]
    g_body = [p for n, p in model.named_parameters() if p.requires_grad and n not in head_names and n not in new_names]
    groups = [dict(params=g_head, lr=J["head_lr"])]
    if g_body:
        groups.append(dict(params=g_body, lr=J["lr"], is_body=True))
    opt = torch.optim.AdamW(groups, weight_decay=J["weight_decay"])
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    steps_per_epoch = math.ceil(len(W) / J["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(0.06 * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)   # low start: fewer skipped fp16 steps
    n_skip = 0
    best = dict(f1=-1, epoch=0, state=None, report=None, pred=None)
    hist, step, bad = [], 0, 0
    t0 = time.time()
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    for ep in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(len(W))
        tl, tn, gsum = 0.0, 0, [0.0]
        for k in range(0, len(W), J["batch_size"]):
            ids, am, y = collate([W[i] for i in perm[k:k + J["batch_size"]]])
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            f = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))   # warmup + linear decay
            # LP-FT: optionally keep the pretrained body frozen for the first epoch(s) so the random head
            # does not distort the Phase-1 features, then re-warm the body learning rate.
            nf = J.get("freeze_body_epochs", 0)
            if ep <= nf:
                fb = 0.0
            else:
                bs = step - nf * steps_per_epoch
                fb = min(1.0, (bs + 1) / warm) * (1.0 if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * (fb if (g.get("is_body") and nf) else f)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lg = model(ids, am)
            lg = lg.float()
            loss = F.cross_entropy(lg.view(-1, lg.size(-1)), y.view(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    tl_ = teacher(ids, am).float()
                Tt = J["kd_temperature"]
                kl = F.kl_div(F.log_softmax(lg / Tt, -1), F.log_softmax(tl_ / Tt, -1), log_target=True, reduction="none").sum(-1)
                kl = (kl * am).sum() / am.sum()
                loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * (Tt ** 2) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            # clip each parameter group separately: a large body gradient must never shrink the head's update
            gn = [float(torch.nn.utils.clip_grad_norm_(g["params"], 1.0)) for g in opt.param_groups]
            gsum = [a + b for a, b in zip(gsum, gn)] if tn else gn
            sc0 = scaler.get_scale() if use_amp else 1.0
            scaler.step(opt); scaler.update()
            if use_amp and scaler.get_scale() < sc0:
                n_skip += 1                                     # overflow -> this optimizer step was skipped
            tl += loss.item(); tn += 1; step += 1
        rec = dict(epoch=ep, train_loss=tl / max(tn, 1), secs=time.time() - t0, skipped_steps=n_skip,
                   grad_norm_head=round(gsum[0] / max(tn, 1), 3),
                   grad_norm_body=round(gsum[1] / max(tn, 1), 3) if len(gsum) > 1 else None)
        if val_docs:
            lps = doc_logprobs(model, val_docs, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
            pred = decode_docs(val_docs, lps, tags)
            sc = strict_scores({d: v["gold"] for d, v in val_docs.items()}, pred, labels)
            rec.update(val_f1=sc["macro_f1"], val_p=sc["macro_p"], val_r=sc["macro_r"])
            if sc["macro_f1"] > best["f1"]:
                best.update(f1=sc["macro_f1"], epoch=ep, report=sc, pred=pred,
                            state={n: p.detach().cpu().clone() for n, p in model.state_dict().items()})
                bad = 0
            else:
                bad += 1
        hist.append(rec)
        log(json.dumps(rec))
        if val_docs and bad >= J["patience"]:
            log(f"early stop at epoch {ep} (best epoch {best['epoch']}, val F1 {best['f1']:.4f})")
            break
    info = dict(train_secs=time.time() - t0, history=hist, n_windows=len(W), steps=step,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2**30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    labels, tags, folds = D["labels"], D["tags"], D["folds"]
    if J.get("quick"):
        folds = {f: dict(list(v.items())[: J["quick_docs"]]) for f, v in folds.items()}
    method = J["method"]
    base_factory = lambda: load_backbone(J)
    log(f"arch={J.get('arch', 'txl')} method={method} attn={J['attn']} head={J['head']} device={device}")

    def get_teacher(which):
        tp = Path(J["kd_teacher_dir"]) / f"{which}_model.pt"
        assert tp.exists(), f"KD needs the teacher checkpoint {tp} (run the teacher job first)"
        tj = json.load(open(Path(J["kd_teacher_dir"]) / "job.json"))
        t = build_model(tj, base_factory, len(tags), tj["method"])
        t.load_state_dict(torch.load(tp, map_location="cpu"))
        t.to(device).eval()
        for p in t.parameters():
            p.requires_grad_(False)
        return t

    results = dict(name=J["name"], method=method, attn=J["attn"], head=J["head"], config=J)
    # ---- (1) model selection: train fold1 -> validate fold2
    model = build_model(J, base_factory, len(tags), method)
    results["params_total"], results["params_trainable"] = count_params(model)
    log(f"params total={results['params_total']/1e6:.2f}M trainable={results['params_trainable']/1e6:.3f}M")
    teacher = get_teacher("val") if method == "kd" else None
    if teacher is not None:
        results["teacher_params"] = count_params(teacher)[0]
    best, info = train_one(J, model, folds["fold1"], folds["fold2"], tags, labels, device, J["epochs"], log, teacher)
    results.update(val_macro_f1=best["f1"], best_epoch=best["epoch"], val_report=best["report"], val_train=info,
                   val_errors=error_breakdown({d: v["gold"] for d, v in folds["fold2"].items()}, best["pred"]))
    json.dump(best["pred"], open(out / "val_pred_fold2.json", "w"))
    log(f"VALIDATION fold1->fold2 strict macro-F1 = {best['f1']:.4f} (epoch {best['epoch']})")
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    del teacher
    # ---- (2) final model
    if J["final_fit"] == "all":
        epochs = max(1, best["epoch"] + J.get("final_extra_epochs", 0))
        model = build_model(J, base_factory, len(tags), method)
        teacher = get_teacher("final") if method == "kd" else None
        train_docs = {**folds["fold1"], **folds["fold2"]}
        _, finfo = train_one(J, model, train_docs, None, tags, labels, device, epochs, log, teacher)
        results["final_train"] = finfo
        del teacher
    else:
        model.load_state_dict(best["state"])
    model.to(device)
    if J["save_model"]:
        torch.save({n: p.detach().cpu() for n, p in model.state_dict().items()}, out / "final_model.pt")
    # ---- (3) test prediction
    t0 = time.time()
    test = folds["fold3"]
    lps = doc_logprobs(model, test, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
    pred = decode_docs(test, lps, tags)
    results["test_infer_secs"] = time.time() - t0
    all_ids = list(D["folds"]["fold3"].keys())
    final = {did: [[int(s), int(e), l] for s, e, l in pred.get(did, [])] for did in all_ids}
    assert set(final) == set(all_ids)
    for did, spans in final.items():
        n = len(D["folds"]["fold3"][did]["text"])
        assert all(0 <= s < e <= n and l in labels for s, e, l in spans), did
    pj = out / "prediction.json"
    json.dump(final, open(pj, "w"))
    zp = out / f"submission_{J['name']}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    results["test_entities"] = sum(len(v) for v in final.values())
    results["test_label_counts"] = dict(Counter(l for v in final.values() for _, _, l in v))
    json.dump(results, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> {zp} | test entities={results['test_entities']} | {json.dumps(results['test_label_counts'])}")


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p6/p2_ner.py


In [3]:
%%writefile /kaggle/working/p6/p6_summ.py
"""CS-781 Phase 6 - Abstractive legal summarization (BERTScore F1; ROUGE-1 / ROUGE-L) on my Phase-1 models.

One file, used two ways (same as Phases 2-5):
  * imported by the notebook  -> data cleaning, tokenisation checks, sentence units, learned context selector
  * run as a script            -> `python p6_summ.py job.json` trains / evaluates / generates ONE experiment

Data facts that shaped the design (Phase_6_Summ, checked again in the notebook):
  * 7,030 train documents with reference summaries, 100 test documents, no dev split (we hold out 400 train docs).
  * Documents: median ~3.1k words (p90 ~7.9k, max 140k). Summaries: median ~640 words (p90 1.5k), ~22% of the
    document, fairly extractive (only 4% / 22% / 37% of summary 1/2/3-grams are not in the document).
  * BERTScore (roberta-large) truncates every text to 512 roberta tokens (~400 words): only the START of the
    reference summary is compared -> we generate ~`sum_tokens` tokens (~390 words) and train on the summary start.

Long documents: EXTRACT-THEN-ABSTRACT (importance-based context selection).
  The document is split into sentence units. A small gradient-boosting regressor (sklearn, trained only on the
  Phase-6 train split, no pretrained model) predicts how much of each unit re-appears in the reference summary
  (label = share of the unit's word bigrams that occur in the summary start). The highest-scoring units that fit
  the model's context budget are kept in document order. Alternatives for the report: head / tail / head+tail /
  TF-IDF centrality, and an oracle upper bound (the cell-4 table measures how much of the reference each keeps).

Decoder-only (arch=dec, 1024 RoPE positions): [soft prompt][selected context] " Summary:" [summary ... </s>]
  in ONE causal sequence (the exact Phase-1 computation), loss only on the summary tokens.
  Generation: KV cache, batched (left padding), n-gram blocking / repetition penalty, decoding chosen on dev.

Adaptation methods (Phase-6 list: Houlsby / Pfeiffer adapters, prefix, prompt, LoRA, QLoRA, AdaLoRA, LoRMA, KD):
  lorma      LoRMA_+ (Bihany, Patel & Modi 2025): y = (I + s*B A) W x         (compulsory; KD teacher candidate)
  lora       LoRA: y = W x + s*B A x
  qlora      frozen weights in 4-bit NF4 (block 64, double-quantised scales) + LoRA
  adalora    AdaLoRA (peft)
  adapter    Houlsby (after attention output + FFN output) | Pfeiffer (after the FFN output only)
  prompt     prompt tuning: n_prompt trainable input embeddings
  prefix     prefix tuning (P-Tuning-v2 form): trainable vectors at the input of EVERY layer's attention
  kd         knowledge distillation: best teacher (lorma / full) -> shallower student built from Phase-1 layers;
             loss = a * CE(reference) + (1-a) * T^2 * KL(teacher || student) on every summary token
  full       full fine-tuning (reference only: not in the Phase-6 list; KD teacher candidate)
  baselines  no training: zero-shot Phase-1 model, extractive selector, lead (for the analysis)
"""
import hashlib
import json
import math
import multiprocessing as mp
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parent))
import p2_ner as P2                            # noqa: E402  (Phase-2 loaders / tokenizers / LoRA / LoRMA / adapters)


# ============================================================== data: cleaning + held-out dev split
def _md5(t):
    return hashlib.md5(t.encode("utf-8", "ignore")).hexdigest()


def _txt(v):
    return " ".join(map(str, v)) if isinstance(v, list) else str(v if v is not None else "")


def _read(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, dict):
        d = [dict(id=k, **v) if isinstance(v, dict) else dict(id=k, document=v) for k, v in d.items()]
    out = []
    for x in d:
        doc = _txt(x.get("document", x.get("text", x.get("judgment"))))
        s = x.get("summary", x.get("reference"))
        out.append(dict(id=x["id"], document=doc, summary=None if s is None else _txt(s)))
    return out


def load_summ(data_dir, n_dev=400, seed=781, log=print):
    """-> dict(train, dev, test, stats). Test summaries are never read. Train rows identical to a test document,
    duplicate documents and empty documents / summaries are dropped; `n_dev` random train documents become dev."""
    data_dir = Path(data_dir)
    test = _read(data_dir / "Test" / "test.json")
    for x in test:
        x["summary"] = None
    tid = [str(x["id"]) for x in test]
    assert len(set(tid)) == len(tid), "duplicate ids in test.json"
    tid, th = set(tid), {_md5(x["document"]) for x in test}
    raw = _read(data_dir / "Train" / "train.json")
    st, seen, keep = Counter(), set(), []
    for x in raw:
        h = _md5(x["document"])
        if str(x["id"]) in tid or h in th:
            st["dropped_same_id_or_text_as_test"] += 1
        elif not x["document"].strip() or not (x["summary"] or "").strip():
            st["dropped_empty_document_or_summary"] += 1
        elif h in seen:
            st["dropped_duplicate_document"] += 1
        else:
            seen.add(h)
            keep.append(x)
    perm = np.random.default_rng(seed).permutation(len(keep))
    dev = [keep[i] for i in sorted(perm[:n_dev])]
    train = [keep[i] for i in sorted(perm[n_dev:])]
    stats = dict(train_rows=len(raw), kept=len(keep), train=len(train), dev=len(dev), test=len(test), cleaning=dict(st))
    log(json.dumps(stats))
    return dict(train=train, dev=dev, test=test, stats=stats)


# ============================================================== text pipelines (exact Phase-1 preprocessing)
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")


def txl_normalize(t):
    """Exactly prep_data.normalize_text of the Phase-1 Transformer-XL runs."""
    if not t:
        return ""
    if not t.isascii():
        t = t.translate(_MAP)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return " ".join(t.split())


def text_fns(tok, arch):
    """(normalize(raw) -> the text Phase-1 pretraining tokenised,  encode(normalized text) -> ids)."""
    if tok.kind == "hf":
        return (lambda t: " ".join(t.split())), (lambda s: tok.tok.encode(s).ids)
    if arch == "txl" and tok.kind == "old":
        enc = tok.tok._encode_pretoken
        return txl_normalize, (lambda s: [i for pc in P2.PRE_RE.findall(s) for i in enc(pc)])
    if arch == "txl":
        return txl_normalize, (lambda s: tok.tok.encode(s, normalize=False))
    if tok.kind == "new":
        return tok.mod.normalize_text, (lambda s: tok.tok.encode(s, normalize=False))
    if tok.norm == "clean":
        enc = tok.tok._encode_pretoken
        return P2.clean_text_ret, (lambda s: [i for pc in P2.PRE_RE_ORIG.findall(s) for i in enc(pc)])
    raise ValueError(f"no Phase-1 text pipeline for arch={arch} tokenizer={tok.kind}/{tok.norm}")


def special_ids(tok):
    if tok.kind == "new":
        return dict(pad=tok.mod.PAD_ID, bos=tok.mod.BOS_ID, eos=tok.mod.EOS_ID, unk=tok.mod.UNK_ID)
    if tok.kind == "old":
        return dict(pad=0, unk=1, bos=2, eos=3)
    return dict(pad=0, bos=1, eos=2, unk=3)


def decode_fn(tok):
    if tok.kind == "hf":
        return lambda ids: tok.tok.decode([int(i) for i in ids if int(i) > 3])
    if tok.kind == "new":
        return lambda ids: tok.tok.decode(ids)
    return lambda ids: tok.tok.decode([int(i) for i in ids if int(i) > 3])


# ============================================================== sentence units (whitespace-aligned = identical ids)
_SENT_WS = re.compile(r'(?<=[.?!;:])\s+(?=[A-Z0-9("\'\[])')
_WS = re.compile(r"\s+")
_WORD = re.compile(r"[a-z0-9]+")


def split_units(t, min_words=6, max_words=70):
    """Normalised text -> list of EXACT substrings ("".join(units) == t). Every unit after the first starts with its
    whitespace, so encoding unit by unit gives exactly the ids of the whole text (asserted in the notebook)."""
    cuts = [0] + [m.start() for m in _SENT_WS.finditer(t)] + [len(t)]
    pieces = [t[a:b] for a, b in zip(cuts, cuts[1:]) if b > a]
    out = []
    for p in pieces:
        nw = len(p.split())
        if out and nw < min_words:
            out[-1] += p
            continue
        while nw > max_words:
            ends = [m.start() for m in _WS.finditer(p) if m.start() > 0]
            c = ends[max_words - 1]
            out.append(p[:c])
            p = p[c:]
            nw = len(p.split())
        out.append(p)
    return out


CUES = ("appellant", "respondent", "petitioner", "accused", "held", "contended", "submitted", "section", "act",
        "court", "high", "tribunal", "dismissed", "allowed", "filed", "order", "facts", "therefore", "question", "notice")
_JS_RE = re.compile(r"(?:was delivered by|delivered the [Jj]udgment|\bJUDGMENT\b|\bJ\.\s*:)")
_CAP = re.compile(r"[A-Za-z]+")
N_FEAT = 13 + len(CUES)
F_COS = 7                                                    # column of the TF-IDF centrality feature


def unit_features(units, idf, idf_default):
    """Per-unit features for the context selector (position, length, header-ness, TF-IDF centrality, cue words)."""
    n = len(units)
    toks = [_WORD.findall(u.lower()) for u in units]
    dtf = Counter(w for t in toks for w in t)
    D = {w: c * idf.get(w, idf_default) for w, c in dtf.items()}
    Dn = math.sqrt(sum(v * v for v in D.values())) or 1.0
    js = next((k for k, u in enumerate(units) if _JS_RE.search(u)), 0)
    Fm = np.zeros((n, N_FEAT), np.float32)
    for k, (u, t) in enumerate(zip(units, toks)):
        nw = len(u.split())
        tf = Counter(t)
        wv = {w: c * idf.get(w, idf_default) for w, c in tf.items()}
        un = math.sqrt(sum(v * v for v in wv.values()))
        cos = sum(v * D[w] for w, v in wv.items()) / (un * Dn) if un else 0.0
        caps = _CAP.findall(u)
        Fm[k, :13] = [k / max(1, n - 1), min(k, 300), min(n - 1 - k, 300), nw, math.log1p(nw),
                      sum(w.isdigit() for w in t) / max(1, len(t)), sum(w[0].isupper() for w in caps) / max(1, len(caps)),
                      cos, float(np.mean([idf.get(w, idf_default) for w in t])) if t else 0.0, float(k >= js),
                      float(np.clip(k - js, -50, 300)), js / max(1, n), min(n, 2000)]
        Fm[k, 13:] = [tf[c] for c in CUES]
    return Fm, toks


def unit_labels(toks, summary, prefix_words):
    """Share of each unit's word bigrams that occur in the START of the reference summary (what BERTScore sees)."""
    st = _WORD.findall(" ".join(summary.split()[:prefix_words]).lower())
    sb = set(zip(st, st[1:]))
    lab = np.zeros(len(toks), np.float32)
    for k, t in enumerate(toks):
        bg = list(zip(t, t[1:]))
        if bg:
            lab[k] = sum(b in sb for b in bg) / len(bg)
    return lab


def compute_idf(docs, norm_fn, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(docs), min(n, len(docs)), replace=False)
    df = Counter()
    for i in idx:
        df.update(set(_WORD.findall(norm_fn(docs[i]["document"]).lower())))
    N = len(idx)
    idf = {w: math.log((N + 1) / (c + 1)) + 1.0 for w, c in df.items() if c >= 2}
    return idf, math.log(N + 1) + 1.0


_W = {}                                                      # per-worker state (fork)


def _init_worker(spec):
    tok = P2.load_bpe(spec["tok_py"], spec["tok_path"], norm=spec["norm"])
    _W["norm"], _W["enc"] = text_fns(tok, spec["arch"])
    _W["spec"] = spec


def _work(i):
    x, s = _W["items"][i], _W["spec"]
    t = _W["norm"](x["document"])
    units = split_units(t) if t else [""]
    uids = [np.asarray(_W["enc"](u), np.uint16) for u in units]
    feats, toks = unit_features(units, _W["idf"], _W["idf_default"])
    out = dict(id=x["id"], units=units, uids=uids, feats=feats, n_words=len(t.split()), n_tok=int(sum(len(a) for a in uids)))
    if x["summary"] is not None:
        sm = _W["norm"](x["summary"])
        out["sum_ids"] = np.asarray(_W["enc"](s.get("sum_prefix", " ") + sm), np.uint16)
        out["labels"] = unit_labels(toks, sm, s["label_prefix_words"])
        out["ref"] = x["summary"]                            # raw reference (what the evaluator compares with)
        out["ref_prefix_words"] = sm.split()[:s["label_prefix_words"]]
    return i, out


def prepare(data, spec, n_proc=None, log=print):
    """Tokenise + split into units + selector features for every split (multiprocessing, fork)."""
    n_proc = n_proc or max(1, os.cpu_count() or 1)
    tok = P2.load_bpe(spec["tok_py"], spec["tok_path"], norm=spec["norm"])
    norm_fn, _ = text_fns(tok, spec["arch"])
    _W["idf"], _W["idf_default"] = compute_idf(data["train"], norm_fn)
    splits = {}
    for sp in ("train", "dev", "test"):
        items = data[sp]
        _W["items"] = items
        out = [None] * len(items)
        t0 = time.time()
        if n_proc > 1:
            with mp.get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(spec,)) as pool:
                for i, r in pool.imap_unordered(_work, range(len(items)), chunksize=8):
                    out[i] = r
        else:
            _init_worker(spec)
            for i in range(len(items)):
                out[i] = _work(i)[1]
        splits[sp] = out
        log(f"[prepare] {sp}: {len(items)} documents in {time.time() - t0:.0f}s")
    _W.pop("items", None)
    return splits


# ============================================================== context selection (long documents)
def train_selector(train_docs, seed=0, max_rows=400_000, log=print):
    from sklearn.ensemble import HistGradientBoostingRegressor
    X = np.concatenate([d["feats"] for d in train_docs])
    y = np.concatenate([d["labels"] for d in train_docs])
    if len(X) > max_rows:
        idx = np.random.default_rng(seed).choice(len(X), max_rows, replace=False)
        X, y = X[idx], y[idx]
    t0 = time.time()
    m = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=40, random_state=seed)
    m.fit(X, y)
    log(f"[selector] trained on {len(X)} sentence units in {time.time() - t0:.0f}s")
    return m


def score_units(docs, selector):
    for d in docs:
        d["s_central"] = d["feats"][:, F_COS].copy()
        d["s_learned"] = selector.predict(d["feats"]).astype(np.float32) if selector is not None else d["s_central"]
        if "labels" in d:
            d["s_oracle"] = d["labels"]


def select_units(d, budget, mode):
    """-> list of (unit index, token slice) in document order whose ids fit in `budget` tokens."""
    lens = [len(a) for a in d["uids"]]
    n = len(lens)
    if mode in ("head", "tail", "headtail"):
        hb = budget if mode == "head" else (0 if mode == "tail" else budget // 2)
        out, used = [], 0
        for k in range(n):                                   # head part (last unit cut to fit)
            if used >= hb:
                break
            take = min(lens[k], hb - used)
            out.append((k, slice(0, take)))
            used += take
        last = out[-1][0] if out else -1
        tail, tb = [], budget - used
        for k in range(n - 1, last, -1):                     # tail part (first unit cut from its start)
            if tb <= 0:
                break
            take = min(lens[k], tb)
            tail.append((k, slice(lens[k] - take, lens[k])))
            tb -= take
        return out + tail[::-1]
    sc = d["s_" + mode]
    chosen, used = [], 0
    for k in np.argsort(-sc, kind="stable"):
        if used + lens[k] <= budget:
            chosen.append(int(k))
            used += lens[k]
            if budget - used < 8:
                break
    if not chosen:
        return select_units(d, budget, "head")
    return [(k, slice(0, lens[k])) for k in sorted(chosen)]


def build_context(d, budget, mode):
    sel = select_units(d, budget, mode)
    ids = np.concatenate([d["uids"][k][sl] for k, sl in sel]) if sel else np.zeros(0, np.uint16)
    return ids[:budget].astype(np.int64)


def context_text(d, budget, mode):
    return "".join(d["units"][k] for k, _ in select_units(d, budget, mode))


def content_recall(ctx_text, ref_words):
    """Share of the reference's content-word types (len > 3) that appear in the given text."""
    ref = {w for w in _WORD.findall(" ".join(ref_words).lower()) if len(w) > 3}
    got = set(_WORD.findall(ctx_text.lower()))
    return len(ref & got) / max(1, len(ref))


def coverage_table(docs, budget, modes=("head", "tail", "headtail", "central", "learned", "oracle")):
    """How much of the reference summary start is even VISIBLE to the model under each long-document strategy."""
    rows = {}
    for m in modes:
        r = [content_recall(context_text(d, budget, m), d["ref_prefix_words"]) for d in docs]
        rows[m] = dict(content_recall=round(float(np.mean(r)), 4))
    full = [content_recall("".join(d["units"]), d["ref_prefix_words"]) for d in docs]
    rows["whole document"] = dict(content_recall=round(float(np.mean(full)), 4))
    return rows


# ============================================================== metrics (local proxies of the evaluator)
def words(t):
    return _WORD.findall(t.lower())


def _ngr(w, n):
    return Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))


def rouge_n(c, r, n=1):
    a, b = _ngr(c, n), _ngr(r, n)
    ov = sum((a & b).values())
    if not ov:
        return 0.0, 0.0, 0.0
    p, rc = ov / sum(a.values()), ov / sum(b.values())
    return p, rc, 2 * p * rc / (p + rc)


def lcs_len(a, b):
    """Bit-parallel LCS length (Hyyro 2004): O(len(a) * len(b) / wordsize)."""
    if not a or not b:
        return 0
    M = {}
    for i, t in enumerate(b):
        M[t] = M.get(t, 0) | (1 << i)
    mask = (1 << len(b)) - 1
    V = mask
    for t in a:
        U = V & M.get(t, 0)
        V = ((V + U) | (V - U)) & mask
    return len(b) - bin(V).count("1")


def rouge_l(c, r):
    l = lcs_len(c, r)
    if not l:
        return 0.0
    p, rc = l / len(c), l / len(r)
    return 2 * p * rc / (p + rc)


def rep_ngram(w, n=4):
    g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not g else 1.0 - len(set(g)) / len(g)


_NUM = re.compile(r"\d+")


def pair_metrics(cand, ref, doc_text):
    c, r = words(cand), words(ref)
    d = words(doc_text)
    db = set(zip(d, d[1:]))
    cb = list(zip(c, c[1:]))
    nums = set(_NUM.findall(cand))
    dn = set(_NUM.findall(doc_text))
    return dict(rouge1=rouge_n(c, r, 1)[2], rouge2=rouge_n(c, r, 2)[2], rougeL=rouge_l(c, r), rouge1_recall=rouge_n(c, r, 1)[1],
                words=len(cand.split()), rep4=rep_ngram(c, 4),
                novel_bigrams=(sum(b not in db for b in cb) / len(cb)) if cb else 0.0,
                number_precision=(len(nums & dn) / len(nums)) if nums else float("nan"))


_BS = {}


def bertscore(cands, refs, device="cuda", batch_size=16):
    """BERTScore (roberta-large, layer 17, no baseline rescaling = the bert_score default the evaluator uses).
    Used ONLY to measure / choose settings on our own dev split. -> (P, R, F) numpy arrays or None if unavailable."""
    if "s" not in _BS:
        try:
            from bert_score import BERTScorer
            _BS["s"] = BERTScorer(model_type="roberta-large", lang="en", device=str(device), batch_size=batch_size)
        except Exception as e:                               # no internet / package -> ROUGE is used instead
            _BS["s"], _BS["err"] = None, repr(e)[:200]
    if _BS["s"] is None:
        return None
    c = [x if x.strip() else "." for x in cands]
    P, R, F1 = _BS["s"].score(c, list(refs))
    return P.numpy(), R.numpy(), F1.numpy()


def score_outputs(cands, refs, docs_text, device, with_bert=True):
    pm = [pair_metrics(c, r, d) for c, r, d in zip(cands, refs, docs_text)]
    res = {k: float(np.nanmean([m[k] for m in pm])) if any(not math.isnan(m[k]) for m in pm) else float("nan") for k in pm[0]}
    res["per_doc"] = pm
    bs = bertscore(cands, refs, device) if with_bert else None
    if bs is not None:
        res["bertscore_p"], res["bertscore_r"], res["bertscore_f1"] = (float(x.mean()) for x in bs)
        for m, f in zip(pm, bs[2]):
            m["bertscore_f1"] = float(f)
    res["key"] = res.get("bertscore_f1", (res["rouge1"] + res["rougeL"]) / 2)
    return res


# ============================================================== QLoRA: NF4 4-bit storage of the frozen weights
NF4_LEVELS = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
              -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
              0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
              0.7229568362236023, 1.0]


class NF4Linear(nn.Module):
    """Frozen nn.Linear stored as 4-bit NormalFloat (QLoRA): blocks of `block` weights share one absmax scale; the
    scales are 8-bit quantised in groups of 256 (double quantisation). De-quantised on the fly while training; for
    generation the de-quantised fp16 matrix is cached once (`cache_dense`), the arithmetic is identical."""

    def __init__(self, lin: nn.Linear, block=64, block2=256):
        super().__init__()
        W = lin.weight.detach().float().cpu()
        self.in_features, self.out_features, self.block = lin.in_features, lin.out_features, block
        self.n = W.numel()
        flat = torch.cat([W.reshape(-1), W.new_zeros((-self.n) % block)]).view(-1, block)
        amax = flat.abs().amax(1).clamp(min=1e-12)
        lv = torch.tensor(NF4_LEVELS)
        codes = torch.bucketize((flat / amax[:, None]).clamp(-1, 1), (lv[1:] + lv[:-1]) / 2).to(torch.uint8)
        self.register_buffer("qweight", (codes[:, 0::2] << 4) | codes[:, 1::2])
        self.nb = amax.numel()
        a = torch.cat([amax, amax.new_zeros((-self.nb) % block2)]).view(-1, block2)
        s2 = a.amax(1).clamp(min=1e-12) / 255.0
        self.register_buffer("absmax_q", torch.round(a / s2[:, None]).to(torch.uint8))
        self.register_buffer("absmax_s2", s2)
        self.register_buffer("levels", lv, persistent=False)
        self.bias = nn.Parameter(lin.bias.detach().clone(), requires_grad=False) if lin.bias is not None else None
        self._dense = None

    def dequant(self, dtype):
        q = self.qweight
        codes = torch.stack([(q >> 4), (q & 15)], -1).view(q.size(0), -1).long()
        amax = (self.absmax_q.float() * self.absmax_s2[:, None]).view(-1)[: self.nb]
        W = (self.levels.to(q.device)[codes] * amax[:, None]).view(-1)[: self.n]
        return W.view(self.out_features, self.in_features).to(dtype)

    def cache_dense(self, dtype):
        self._dense = self.dequant(dtype)

    def forward(self, x):
        dt = torch.float16 if (x.is_cuda and torch.is_autocast_enabled()) else x.dtype
        W = self._dense if (self._dense is not None and not self.training and self._dense.dtype == dt) else self.dequant(dt)
        b = None if self.bias is None else self.bias.to(dt)
        return F.linear(x.to(dt), W, b)

    def nbytes(self):
        return self.qweight.numel() + self.absmax_q.numel() + 4 * self.absmax_s2.numel()


# ============================================================== backend: decoder-only LegalLM (arch=dec)
class DecSumm(nn.Module):
    """Phase-1 LegalLM (RoPE / RMSNorm / SwiGLU) as a causal summariser. Same weights and the same computation as
    Phase 1 (checked in the notebook); adds soft prompts / prefixes and a KV cache that works with every PEFT wrapper."""

    def __init__(self, base, layers=None, dropout=0.1):
        super().__init__()
        self.cfg = base.cfg
        self.tok_emb, self.norm, self.lm_head = base.tok_emb, base.norm, base.lm_head
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.rope = sys.modules[type(base).__module__]
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.norm
        self.max_pos = self.cfg.max_seq_len
        self.hd = self.cfg.d_model // self.cfg.n_head
        self.n_prompt, self.prompt, self.prefix = 0, None, None
        self._ckpt = False

    @property
    def d(self):
        return self.cfg.d_model

    def _block(self, li, x, cos, sin, pp_idx, mask, cache, cache_len):
        blk = self.blocks[li]
        a = blk.attn
        B, T, _ = x.shape
        hx = blk.attn_norm(x)
        if self.prefix is not None and pp_idx is not None:
            rows = torch.arange(B, device=x.device)[:, None]
            hx = hx.index_put((rows, pp_idx), self.prefix[li].to(hx.dtype)[None].expand(B, -1, -1))
        q = a.q_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
        k = a.k_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
        v = a.v_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
        q, k = self.rope.apply_rope(q, cos, sin), self.rope.apply_rope(k, cos, sin)
        if cache is None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=mask is None)
        else:
            ck, cv = cache[li]
            ck[:, :, cache_len:cache_len + T] = k.to(ck.dtype)
            cv[:, :, cache_len:cache_len + T] = v.to(cv.dtype)
            n = cache_len + T
            y = F.scaled_dot_product_attention(q, ck[:, :, :n].to(q.dtype), cv[:, :, :n].to(q.dtype), attn_mask=mask)
        x = x + blk.drop(a.o_proj(y.transpose(1, 2).reshape(B, T, -1)))
        return x + blk.drop(blk.mlp(blk.mlp_norm(x)))

    def hidden(self, ids, pos, pp_idx=None, mask=None, cache=None, cache_len=0):
        """ids/pos [B,T]. mask None -> causal (right padding); else bool [B,1,T,keys] (True = may attend)."""
        B, T = ids.shape
        cos, sin = self.rope.rope_cos_sin(pos, self.hd, self.cfg.rope_theta)
        x = self.tok_emb(ids)
        if self.prompt is not None and pp_idx is not None:
            rows = torch.arange(B, device=ids.device)[:, None]
            x = x.index_put((rows, pp_idx), self.prompt.to(x.dtype)[None].expand(B, -1, -1))
        for li in range(len(self.blocks)):
            if self._ckpt and self.training and cache is None:
                x = torch.utils.checkpoint.checkpoint(self._block, li, x, cos, sin, pp_idx, mask, None, 0, use_reentrant=False)
            else:
                x = self._block(li, x, cos, sin, pp_idx, mask, cache, cache_len)
        return self.norm(x)

    def init_cache(self, B, L, device, dtype):
        return [(torch.zeros(B, self.cfg.n_head, L, self.hd, device=device, dtype=dtype),
                 torch.zeros(B, self.cfg.n_head, L, self.hd, device=device, dtype=dtype)) for _ in self.blocks]

    # ---- prompt / prefix tuning: soft vectors at the first n_prompt positions of every sequence
    def add_prompts(self, n, deep, seed=0):
        g = torch.Generator().manual_seed(seed)
        rows = torch.randint(300, min(20000, self.tok_emb.num_embeddings), (n,), generator=g)
        self.n_prompt = n
        self.prompt = nn.Parameter(self.tok_emb.weight.detach()[rows].clone().float())       # Lester et al.: real tokens
        new = [self.prompt]
        if deep:
            self.prefix = nn.ParameterList()
            for blk in self.blocks:
                w = getattr(blk.attn_norm, "weight", None)
                sc = w.detach().abs().mean().item() if w is not None else 1.0
                self.prefix.append(nn.Parameter(torch.randn(n, self.d, generator=g) * sc))
            new += list(self.prefix)
        return new


class DecBackend:
    arch = "dec"
    lowrank_all = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

    def build(self, J, layers=None):
        base = P2.load_decoder(J["model_py"], J["phase1_ckpt"])[0]
        return DecSumm(base, layers=layers, dropout=J["dropout"])

    def sep_ids(self, J, enc_fn):
        return list(enc_fn(J["sep_text"]))

    def ctx_budget(self, J, n_sep, n_prompt):
        return J["max_len"] - n_prompt - n_sep - J["sum_tokens"] - 1

    def gen_prompt(self, d, J):
        return list(build_context(d, J["_ctx_budget"], J["ctx_select"])) + J["_sep"]

    def train_sample(self, d, J):
        s = d["sum_ids"][:J["sum_tokens"]].astype(np.int64)
        tgt = list(s) + ([J["_eos"]] if len(d["sum_ids"]) <= J["sum_tokens"] else [])
        return self.gen_prompt(d, J), tgt

    def make_batch(self, samples, P, device):
        seqs = [[0] * P + list(c) + list(t) for c, t in samples]
        T = max(len(s) for s in seqs) - 1
        B = len(seqs)
        ids = torch.zeros(B, T, dtype=torch.long)
        lab = torch.full((B, T), -100, dtype=torch.long)
        for b, ((c, t), s) in enumerate(zip(samples, seqs)):
            n = len(s)
            ids[b, :n - 1] = torch.as_tensor(s[:-1])
            st = P + len(c)                                  # first summary token; predicted at position st-1
            lab[b, st - 1:n - 1] = torch.as_tensor(s[st:])
        pos = torch.arange(T)[None].expand(B, T)
        pp = torch.arange(P)[None].expand(B, P).contiguous() if P else None
        to = lambda x: None if x is None else x.to(device)
        return dict(ids=to(ids), lab=to(lab), pos=to(pos), pp=to(pp))

    def loss(self, model, bt, teacher=None, J=None):
        h = model.hidden(bt["ids"], bt["pos"], bt["pp"])
        sel = bt["lab"] != -100
        lg = model.lm_head(h[sel]).float()
        y = bt["lab"][sel]
        loss = F.cross_entropy(lg, y)
        if teacher is not None:
            with torch.no_grad():
                ht = teacher.hidden(bt["ids"], bt["pos"], bt["pp"])
                lt = teacher.lm_head(ht[sel]).float()
            T = J["kd_temperature"]
            kl = F.kl_div(F.log_softmax(lg / T, -1), F.log_softmax(lt / T, -1), log_target=True, reduction="batchmean")
            loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * T * T * kl
        return loss, int(sel.sum())

    @torch.no_grad()
    def generate(self, model, prompts, g, device, ids_cfg):
        """prompts: token lists (context + separator). Returns generated token lists (EOS removed)."""
        model.eval()
        P = model.n_prompt
        B = len(prompts)
        L = P + max(len(p) for p in prompts)
        max_new = int(g["max_new"])
        assert L + max_new <= model.max_pos, f"{L} + {max_new} > {model.max_pos} positions"
        ids = torch.zeros(B, L, dtype=torch.long)
        valid = torch.zeros(B, L, dtype=torch.bool)
        starts = []
        for b, p in enumerate(prompts):
            s0 = L - P - len(p)
            starts.append(s0)
            ids[b, s0 + P:] = torch.as_tensor(p)
            valid[b, s0:] = True
        ids, valid = ids.to(device), valid.to(device)
        pp = (torch.as_tensor(starts, device=device)[:, None] + torch.arange(P, device=device)[None]) if P else None
        pos = (valid.long().cumsum(1) - 1).clamp(min=0)
        nvalid = valid.sum(1)
        causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))
        mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=device)[None, None]
        amp = device.type == "cuda"
        cache = model.init_cache(B, L + max_new, device, torch.float16 if amp else torch.float32)
        key_valid = torch.zeros(B, L + max_new, dtype=torch.bool, device=device)
        key_valid[:, :L] = valid
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            h = model.hidden(ids, pos, pp, mask=mask, cache=cache, cache_len=0)
            logits = model.lm_head(h[:, -1]).float()
        dec = Decoder(B, model.lm_head.weight.shape[0], g, ids_cfg, device)
        for t in range(max_new):
            nxt = dec.step(logits, t)
            if dec.all_done():
                break
            key_valid[:, L + t] = True
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                h = model.hidden(nxt[:, None], (nvalid + t)[:, None], None, mask=key_valid[:, None, None, :L + t + 1],
                                 cache=cache, cache_len=L + t)
                logits = model.lm_head(h[:, -1]).float()
        return dec.outputs()

    @torch.no_grad()
    def check_equivalence(self, base, model, device):
        """(1) causal hidden == Phase-1 hidden_states, (2) KV-cache prefill + steps == full forward (left padding)."""
        model.eval()
        V = model.lm_head.weight.shape[0]
        g = torch.Generator().manual_seed(0)
        ids = torch.randint(300, V, (1, 48), generator=g).to(device)
        d1 = (base.hidden_states(ids) - model.hidden(ids, torch.arange(48, device=device)[None])).abs().max().item()
        a, b = torch.randint(300, V, (40,), generator=g).tolist(), torch.randint(300, V, (25,), generator=g).tolist()
        cont = torch.randint(300, V, (2, 4), generator=g).to(device)
        full = [model.lm_head(model.hidden(torch.as_tensor(p + cont[i].tolist(), device=device)[None],
                                           torch.arange(len(p) + 4, device=device)[None]))[0, -5:] for i, p in enumerate([a, b])]
        L = 40
        ids2 = torch.zeros(2, L, dtype=torch.long, device=device)
        ids2[0], ids2[1, 15:] = torch.as_tensor(a, device=device), torch.as_tensor(b, device=device)
        valid = torch.zeros(2, L, dtype=torch.bool, device=device)
        valid[0], valid[1, 15:] = True, True
        pos = (valid.long().cumsum(1) - 1).clamp(min=0)
        causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))
        mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=device)[None, None]
        cache = model.init_cache(2, L + 4, device, torch.float32)
        outs = [model.lm_head(model.hidden(ids2, pos, None, mask=mask, cache=cache, cache_len=0))[:, -1]]
        kv = torch.zeros(2, L + 4, dtype=torch.bool, device=device)
        kv[:, :L] = valid
        for t in range(4):
            kv[:, L + t] = True
            outs.append(model.lm_head(model.hidden(cont[:, t:t + 1], (valid.sum(1) + t)[:, None], None,
                                                   mask=kv[:, None, None, :L + t + 1], cache=cache, cache_len=L + t))[:, -1])
        got = torch.stack(outs, 1)                            # [2, 5, V]
        d2 = max((got[i] - full[i]).abs().max().item() for i in range(2))
        return d1, d2


# ============================================================== backend: Archit-2-v2 encoder-decoder (arch=encdec2)
class ED2Summ(nn.Module):
    """Phase-1 encoder-decoder 143M (6 enc + 8 dec, d768; encoder 1024 / decoder 256 learned positions) as a seq2seq
    summariser, fed exactly like Phase-1: encoder <- text, decoder <- [<s>] + target (causal), target ends with </s>.
    Prompt / prefix tuning act on the ENCODER (soft input tokens / per-layer prefixes); the decoder keeps its 256 positions."""

    def __init__(self, base, layers=None, dropout=0.1):
        super().__init__()
        enc, dec = base.encoder, base.decoder
        self.enc_emb, self.dec_emb = enc.embedding, dec.embedding
        ke = list(range(len(enc.layers))) if layers is None else list(layers["enc"])
        kd = list(range(len(dec.layers))) if layers is None else list(layers["dec"])
        self.layer_ids = dict(enc=ke, dec=kd)
        self.enc_layers = nn.ModuleList([enc.layers[i] for i in ke])
        self.dec_layers = nn.ModuleList([dec.layers[i] for i in kd])
        self.blocks = nn.ModuleList(list(self.enc_layers) + list(self.dec_layers))   # for PEFT
        self.enc_norm, self.dec_norm, self.lm_head = enc.final_norm, dec.final_norm, base.lm_head
        self.final_norm = self.dec_norm
        for m in list(self.modules()):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.max_src = self.enc_emb.position_embedding.num_embeddings
        self.max_pos = self.dec_emb.position_embedding.num_embeddings
        self.n_prompt, self.prompt, self.prefix = 0, None, None
        self._ckpt = False
        self.d = self.enc_emb.d_model

    def encode(self, ids, am, pp=None):
        B = ids.size(0)
        x = self.enc_emb(ids)
        rows = torch.arange(B, device=ids.device)[:, None]
        if self.prompt is not None and pp is not None:
            x = x.index_put((rows, pp), self.prompt.to(x.dtype)[None].expand(B, -1, -1))
        for li, l in enumerate(self.enc_layers):
            def f(x, li=li, l=l):
                h = l.norm1(x)
                if self.prefix is not None and pp is not None:
                    h = h.index_put((rows, pp), self.prefix[li].to(h.dtype)[None].expand(B, -1, -1))
                x = x + l.dropout1(l.self_attn(h, key_padding_mask=am))
                return x + l.dropout2(l.ffn(l.norm2(x)))
            x = torch.utils.checkpoint.checkpoint(f, x, use_reentrant=False) if (self._ckpt and self.training) else f(x)
        return self.enc_norm(x)

    def decode(self, dec_ids, mem, am):
        y = self.dec_emb(dec_ids)
        for l in self.dec_layers:
            y = torch.utils.checkpoint.checkpoint(l, y, mem, am, use_reentrant=False) if (self._ckpt and self.training) else l(y, mem, am)
        return self.dec_norm(y)

    def cross_kv(self, mem):
        return [(l.cross_attn._split(l.cross_attn.k_proj(mem)), l.cross_attn._split(l.cross_attn.v_proj(mem))) for l in self.dec_layers]

    def dec_step(self, tok, t, caches, ckv, am):
        """One decoding step with a self-attention KV cache (same arithmetic as the Phase-1 DecoderLayer.step)."""
        x = self.dec_emb(tok, offset=t)
        B = x.size(0)
        for i, l in enumerate(self.dec_layers):
            a = l.self_attn
            h = l.norm1(x)
            q, k, v = a._split(a.q_proj(h)), a._split(a.k_proj(h)), a._split(a.v_proj(h))
            if caches[i] is not None:
                k, v = torch.cat([caches[i][0], k], 2), torch.cat([caches[i][1], v], 2)
            caches[i] = (k, v)
            o = F.scaled_dot_product_attention(q, k, v)
            x = x + l.dropout1(a.out_proj(o.transpose(1, 2).contiguous().view(B, 1, -1)))
            c = l.cross_attn
            qc = c._split(c.q_proj(l.norm2(x)))
            o = F.scaled_dot_product_attention(qc, ckv[i][0], ckv[i][1], attn_mask=am[:, None, None, :])
            x = x + l.dropout2(c.out_proj(o.transpose(1, 2).contiguous().view(B, 1, -1)))
            x = x + l.dropout3(l.ffn(l.norm3(x)))
        return self.lm_head(self.dec_norm(x)[:, -1])

    def add_prompts(self, n, deep, seed=0):
        g = torch.Generator().manual_seed(seed)
        te = self.enc_emb.token_embedding
        rows = torch.randint(300, min(20000, te.num_embeddings), (n,), generator=g)
        self.n_prompt = n
        self.prompt = nn.Parameter(te.weight.detach()[rows].clone().float() * math.sqrt(self.d))   # as the embedding layer scales them
        new = [self.prompt]
        if deep:
            self.prefix = nn.ParameterList()
            for l in self.enc_layers:
                sc = l.norm1.weight.detach().abs().mean().item()
                self.prefix.append(nn.Parameter(torch.randn(n, self.d, generator=g) * sc))
            new += list(self.prefix)
        return new


class ED2Backend:
    arch = "encdec2"

    def build(self, J, layers=None):
        return ED2Summ(P2.load_encdec2(J["phase1_ckpt"]), layers=layers, dropout=J["dropout"])

    def sep_ids(self, J, enc_fn):
        return []

    def ctx_budget(self, J, n_sep, n_prompt):
        return J["max_len"] - n_prompt                       # the whole encoder window holds the selected context

    def gen_prompt(self, d, J):
        return list(build_context(d, J["_ctx_budget"], J["ctx_select"]))

    def train_sample(self, d, J):
        s = d["sum_ids"][:J["sum_tokens"]].astype(np.int64)
        return self.gen_prompt(d, J), list(s) + ([J["_eos"]] if len(d["sum_ids"]) <= J["sum_tokens"] else [])

    def _src(self, prompts, P, device):
        B, L = len(prompts), P + max(len(p) for p in prompts)
        ids = torch.zeros(B, L, dtype=torch.long)
        am = torch.zeros(B, L, dtype=torch.bool)
        for b, p in enumerate(prompts):
            ids[b, P:P + len(p)] = torch.as_tensor(p)
            am[b, :P + len(p)] = True
        pp = torch.arange(P)[None].expand(B, P).contiguous() if P else None
        return ids.to(device), am.to(device), (pp.to(device) if P else None)

    def make_batch(self, samples, P, device, bos=1):
        src, am, pp = self._src([c for c, _ in samples], P, device)
        T = max(len(t) for _, t in samples)
        dec = torch.zeros(len(samples), T, dtype=torch.long)
        lab = torch.full((len(samples), T), -100, dtype=torch.long)
        for b, (_, t) in enumerate(samples):
            dec[b, :len(t)] = torch.as_tensor([bos] + list(t[:-1]))
            lab[b, :len(t)] = torch.as_tensor(t)
        return dict(src=src, am=am, pp=pp, dec=dec.to(device), lab=lab.to(device))

    def loss(self, model, bt, teacher=None, J=None):
        h = model.decode(bt["dec"], model.encode(bt["src"], bt["am"], bt["pp"]), bt["am"])
        sel = bt["lab"] != -100
        lg = model.lm_head(h[sel]).float()
        loss = F.cross_entropy(lg, bt["lab"][sel])
        if teacher is not None:
            with torch.no_grad():
                ht = teacher.decode(bt["dec"], teacher.encode(bt["src"], bt["am"], bt["pp"]), bt["am"])
                lt = teacher.lm_head(ht[sel]).float()
            T = J["kd_temperature"]
            kl = F.kl_div(F.log_softmax(lg / T, -1), F.log_softmax(lt / T, -1), log_target=True, reduction="batchmean")
            loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * T * T * kl
        return loss, int(sel.sum())

    @torch.no_grad()
    def generate(self, model, prompts, g, device, ids_cfg):
        model.eval()
        max_new = int(g["max_new"])
        assert max_new <= model.max_pos, f"{max_new} > {model.max_pos} decoder positions"
        src, am, pp = self._src(prompts, model.n_prompt, device)
        assert src.size(1) <= model.max_src
        amp = device.type == "cuda"
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            mem = model.encode(src, am, pp)
            ckv = model.cross_kv(mem)
            caches = [None] * len(model.dec_layers)
            tok = torch.full((len(prompts), 1), ids_cfg["bos"], dtype=torch.long, device=device)
            dec = Decoder(len(prompts), model.lm_head.weight.shape[0], g, ids_cfg, device)
            for t in range(max_new):
                logits = model.dec_step(tok, t, caches, ckv, am).float()
                nxt = dec.step(logits, t)
                if dec.all_done():
                    break
                tok = nxt[:, None]
        return dec.outputs()

    @torch.no_grad()
    def check_equivalence(self, base, model, device):
        """(1) teacher-forced logits == the Phase-1 encode + decode_hidden, (2) KV-cache steps == full decoder pass."""
        model.eval(); base.eval()
        V = model.lm_head.weight.shape[0]
        g = torch.Generator().manual_seed(0)
        src = torch.randint(300, V, (2, 40), generator=g).to(device)
        am = torch.ones(2, 40, dtype=torch.bool, device=device)
        am[1, 30:] = False
        src[1, 30:] = 0
        dec = torch.randint(300, V, (2, 12), generator=g).to(device)
        dec[:, 0] = 1
        ref = base.lm_head(base.decode_hidden(dec, base.encode(src, am), am))
        mem = model.encode(src, am)
        mine = model.lm_head(model.decode(dec, mem, am))
        d1 = (ref - mine).abs().max().item()
        ckv, caches = model.cross_kv(mem), [None] * len(model.dec_layers)
        steps = torch.stack([model.dec_step(dec[:, t:t + 1], t, caches, ckv, am) for t in range(12)], 1)
        return d1, (steps - mine).abs().max().item()


# ============================================================== backend: encoder-decoder v2 90M, new BPE (arch=encdec)
class TiedHead(nn.Module):
    """Output layer tied to the shared token embedding (logits = h @ E^T), as in the Phase-1 EncDecLM."""

    def __init__(self, emb):
        super().__init__()
        self.emb = emb

    @property
    def weight(self):
        return self.emb.weight

    def forward(self, h):
        return F.linear(h, self.emb.weight)


class EDNewSumm(nn.Module):
    """Phase-1 EncDecLM 90M (6 + 6 layers, d640; encoder 1024 / decoder 352 learned positions) as a seq2seq summariser:
    encoder <- selected context, decoder <- [<s>] + summary (causal + cross-attention), target ends with </s>.
    Same modules and arithmetic as Phase 1 (checked in the notebook). Prompt / prefix tuning act on the encoder."""

    def __init__(self, base, layers=None, dropout=0.1):
        super().__init__()
        self.c = base.c
        self.tok, self.enc_pos, self.dec_pos, self.drop = base.tok, base.enc_pos, base.dec_pos, base.drop
        ke = list(range(len(base.enc_layers))) if layers is None else list(layers["enc"])
        kd = list(range(len(base.dec_layers))) if layers is None else list(layers["dec"])
        self.layer_ids = dict(enc=ke, dec=kd)
        self.enc_layers = nn.ModuleList([base.enc_layers[i] for i in ke])
        self.dec_layers = nn.ModuleList([base.dec_layers[i] for i in kd])
        self.blocks = nn.ModuleList(list(self.enc_layers) + list(self.dec_layers))   # for PEFT
        self.enc_norm, self.dec_norm = base.enc_norm, base.dec_norm
        self.final_norm = self.dec_norm
        self.lm_head = TiedHead(self.tok)
        for m in list(self.modules()):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.max_src, self.max_pos, self.d = self.c.enc_max_len, self.c.dec_max_len, self.c.d_model
        self.n_prompt, self.prompt, self.prefix = 0, None, None
        self._ckpt = False

    def encode(self, ids, am, pp=None):
        B, T = ids.shape
        x = self.drop(self.tok(ids) + self.enc_pos(torch.arange(T, device=ids.device))[None])
        rows = torch.arange(B, device=ids.device)[:, None]
        if self.prompt is not None and pp is not None:
            x = x.index_put((rows, pp), self.prompt.to(x.dtype)[None].expand(B, -1, -1))
        mask = am[:, None, None, :]
        for li, l in enumerate(self.enc_layers):
            def f(x, li=li, l=l):
                h = l.n1(x)
                if self.prefix is not None and pp is not None:
                    h = h.index_put((rows, pp), self.prefix[li].to(h.dtype)[None].expand(B, -1, -1))
                x = x + l.drop(l.attn(h, mask=mask))
                return x + l.drop(l.ffn(l.n2(x)))
            x = torch.utils.checkpoint.checkpoint(f, x, use_reentrant=False) if (self._ckpt and self.training) else f(x)
        return self.enc_norm(x)

    def decode(self, dec_ids, mem, am):
        T = dec_ids.size(1)
        x = self.drop(self.tok(dec_ids) + self.dec_pos(torch.arange(T, device=dec_ids.device))[None])
        mask = am[:, None, None, :]
        for l in self.dec_layers:
            x = torch.utils.checkpoint.checkpoint(l, x, mem, mask, use_reentrant=False) if (self._ckpt and self.training) else l(x, mem, mask)
        return self.dec_norm(x)

    def cross_kv(self, mem):
        return [l.cross.kv(mem) for l in self.dec_layers]

    def dec_step(self, tok, t, caches, ckv, am):
        x = self.drop(self.tok(tok) + self.dec_pos(torch.arange(t, t + 1, device=tok.device))[None])
        mask = am[:, None, None, :]
        for i, l in enumerate(self.dec_layers):
            a = l.self_attn
            h = l.n1(x)
            q = a.split(a.q(h))
            k, v = a.kv(h)
            if caches[i] is not None:
                k, v = torch.cat([caches[i][0], k], 2), torch.cat([caches[i][1], v], 2)
            caches[i] = (k, v)
            x = x + l.drop(a.attend(q, k, v))
            qc = l.cross.split(l.cross.q(l.n2(x)))
            x = x + l.drop(l.cross.attend(qc, ckv[i][0], ckv[i][1], mask=mask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.lm_head(self.dec_norm(x)[:, -1])

    def add_prompts(self, n, deep, seed=0):
        g = torch.Generator().manual_seed(seed)
        rows = torch.randint(300, min(20000, self.tok.num_embeddings), (n,), generator=g)
        self.n_prompt = n
        self.prompt = nn.Parameter(self.tok.weight.detach()[rows].clone().float())
        new = [self.prompt]
        if deep:
            self.prefix = nn.ParameterList()
            for l in self.enc_layers:
                sc = l.n1.weight.detach().abs().mean().item()
                self.prefix.append(nn.Parameter(torch.randn(n, self.d, generator=g) * sc))
            new += list(self.prefix)
        return new


class EDNewBackend(ED2Backend):
    arch = "encdec"

    def build(self, J, layers=None):
        return EDNewSumm(P2.load_encdec(J["model_py"], J["phase1_ckpt"])[0], layers=layers, dropout=J["dropout"])

    @torch.no_grad()
    def check_equivalence(self, base, model, device):
        """(1) teacher-forced logits == the Phase-1 encode + decode_hidden, (2) KV-cache steps == full decoder pass."""
        model.eval(); base.eval()
        V = model.lm_head.weight.shape[0]
        g = torch.Generator().manual_seed(0)
        src = torch.randint(300, V, (2, 40), generator=g).to(device)
        src[1, 30:] = 0
        dec = torch.randint(300, V, (2, 12), generator=g).to(device)
        dec[:, 0] = 1
        mem0, m0 = base.encode(src)
        ref = base.logits(base.decode_hidden(dec, mem0, m0))
        am = src != 0
        mem = model.encode(src, am)
        mine = model.lm_head(model.decode(dec, mem, am))
        d1 = (ref - mine).abs().max().item()
        ckv, caches = model.cross_kv(mem), [None] * len(model.dec_layers)
        steps = torch.stack([model.dec_step(dec[:, t:t + 1], t, caches, ckv, am) for t in range(12)], 1)
        return d1, (steps - mine).abs().max().item()


# ============================================================== backend: Transformer-XL (arch=txl)
class TXLHead(nn.Module):
    def __init__(self, emb, bias):
        super().__init__()
        self.emb, self.bias = emb, bias

    @property
    def weight(self):
        return self.emb.weight

    def forward(self, h):
        return F.linear(h, self.emb.weight, self.bias)


class TXLSumm(nn.Module):
    """Phase-1 Transformer-XL (relative positions + u/v biases) as a causal summariser over ONE sequence
    [soft prompt][context] " Summary:" [summary]. Relative distances are clamped at seg_len + mem_len - 1 = 511, the
    largest distance the model saw in Phase-1 pretraining (clamp_len of the TXL paper), so every token can still attend
    to the whole context. Same arithmetic as the Phase-1 forward for distances <= 511 (checked in the notebook).
    Generation uses a KV cache (k/v of every layer = the TXL memory)."""

    def __init__(self, base, layers=None, dropout=0.1, clamp=None):
        super().__init__()
        self.cfg = base.cfg
        self.emb, self.pos_emb, self.ln_f, self.drop = base.emb, base.pos_emb, base.ln_f, base.drop
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.final_norm = self.ln_f
        self.lm_head = TXLHead(base.emb, base.out_bias)
        for m in list(self.modules()):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.drop.p = dropout
        self.clamp = clamp if clamp is not None else self.cfg.seg_len + self.cfg.mem_len - 1
        self.max_pos = 10 ** 9                               # relative positions: no hard limit
        self.H, self.dh, self.d = self.cfg.n_head, self.cfg.d_head, self.cfg.d_model
        self.n_prompt, self.prompt, self.prefix = 0, None, None
        self._ckpt = False

    def _block(self, li, h, rs, dist, allow, pp_idx, cache, cache_len):
        blk = self.blocks[li]
        a = blk.attn
        B, T, _ = h.shape
        H, dh = self.H, self.dh
        x = blk.ln1(h)
        if self.prefix is not None and pp_idx is not None:
            rows = torch.arange(B, device=h.device)[:, None]
            x = x.index_put((rows, pp_idx), self.prefix[li].to(x.dtype)[None].expand(B, -1, -1))
        q = a.q_net(x).view(B, T, H, dh).transpose(1, 2)
        kv = a.kv_net(x).view(B, T, 2, H, dh)
        k, v = kv[:, :, 0].transpose(1, 2), kv[:, :, 1].transpose(1, 2)
        if cache is not None:
            ck, cv = cache[li]
            ck[:, :, cache_len:cache_len + T] = k.to(ck.dtype)
            cv[:, :, cache_len:cache_len + T] = v.to(cv.dtype)
            k, v = ck[:, :, :cache_len + T].to(q.dtype), cv[:, :, :cache_len + T].to(q.dtype)
        K, D = k.size(2), rs.size(0)
        rk = a.r_net(rs).view(D, H, dh).permute(1, 0, 2).unsqueeze(0)          # [1,H,D,dh] for distances 0..D-1
        qu = (q + a.u[None, :, None, :]) * a.scale
        qv = (q + a.v[None, :, None, :]) * a.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                               # content term   [B,H,T,K]
        BD = torch.matmul(qv, rk.transpose(-1, -2)).gather(-1, dist[None, None].expand(B, H, T, K))   # position term
        score = (AC + BD).float().masked_fill(~allow, -1e30)
        prob = a.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, T, H * dh)
        h = h + a.drop(a.o_net(out))
        return h + blk.drop(blk.ff2(blk.drop(F.gelu(blk.ff1(blk.ln2(h))))))

    def hidden(self, ids, pos=None, pp_idx=None, mask=None, cache=None, cache_len=0):
        """ids [B,T]; mask None -> causal (right padding), else bool [B,1,T,keys] (True = may attend)."""
        B, T = ids.shape
        dev = ids.device
        h = self.drop(self.emb(ids))
        if self.prompt is not None and pp_idx is not None:
            rows = torch.arange(B, device=dev)[:, None]
            h = h.index_put((rows, pp_idx), self.prompt.to(h.dtype)[None].expand(B, -1, -1))
        K = cache_len + T
        D = min(K, self.clamp + 1)
        qi = torch.arange(cache_len, K, device=dev)[:, None]
        kj = torch.arange(K, device=dev)[None]
        dist = (qi - kj).clamp(0, D - 1)
        allow = mask if mask is not None else (kj <= qi)[None, None]
        rs = self.pos_emb(torch.arange(D, device=dev, dtype=torch.float32))
        for li in range(len(self.blocks)):
            if self._ckpt and self.training and cache is None:
                h = torch.utils.checkpoint.checkpoint(self._block, li, h, rs, dist, allow, pp_idx, None, 0, use_reentrant=False)
            else:
                h = self._block(li, h, rs, dist, allow, pp_idx, cache, cache_len)
        return self.ln_f(h)

    def init_cache(self, B, L, device, dtype):
        return [(torch.zeros(B, self.H, L, self.dh, device=device, dtype=dtype),
                 torch.zeros(B, self.H, L, self.dh, device=device, dtype=dtype)) for _ in self.blocks]

    def add_prompts(self, n, deep, seed=0):
        g = torch.Generator().manual_seed(seed)
        rows = torch.randint(300, min(20000, self.emb.num_embeddings), (n,), generator=g)
        self.n_prompt = n
        self.prompt = nn.Parameter(self.emb.weight.detach()[rows].clone().float())
        new = [self.prompt]
        if deep:
            self.prefix = nn.ParameterList()
            for blk in self.blocks:
                sc = blk.ln1.weight.detach().abs().mean().item()
                self.prefix.append(nn.Parameter(torch.randn(n, self.d, generator=g) * sc))
            new += list(self.prefix)
        return new


class TXLBackend(DecBackend):
    arch = "txl"

    def build(self, J, layers=None):
        base = P2.load_phase1(J["model_py"], J["phase1_ckpt"])[0]
        m = TXLSumm(base, layers=layers, dropout=J["dropout"], clamp=J.get("txl_clamp"))
        m._ckpt = bool(J.get("grad_ckpt", False))
        return m

    @torch.no_grad()
    def check_equivalence(self, base, model, device):
        """(1) logits == the Phase-1 forward (distances < clamp), (2) KV-cache prefill + steps == full forward (left padding)."""
        model.eval(); base.eval()
        V = model.lm_head.weight.shape[0]
        g = torch.Generator().manual_seed(0)
        ids = torch.randint(300, V, (1, 48), generator=g).to(device)
        d1 = (base(ids)["logits"] - model.lm_head(model.hidden(ids))).abs().max().item()
        a, b = torch.randint(300, V, (40,), generator=g).tolist(), torch.randint(300, V, (25,), generator=g).tolist()
        cont = torch.randint(300, V, (2, 4), generator=g).to(device)
        full = [model.lm_head(model.hidden(torch.as_tensor(p + cont[i].tolist(), device=device)[None]))[0, -5:] for i, p in enumerate([a, b])]
        L = 40
        ids2 = torch.zeros(2, L, dtype=torch.long, device=device)
        ids2[0], ids2[1, 15:] = torch.as_tensor(a, device=device), torch.as_tensor(b, device=device)
        valid = torch.zeros(2, L, dtype=torch.bool, device=device)
        valid[0], valid[1, 15:] = True, True
        causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))
        mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=device)[None, None]
        cache = model.init_cache(2, L + 4, device, torch.float32)
        outs = [model.lm_head(model.hidden(ids2, None, None, mask=mask, cache=cache, cache_len=0))[:, -1]]
        kv = torch.zeros(2, L + 4, dtype=torch.bool, device=device)
        kv[:, :L] = valid
        for t in range(4):
            kv[:, L + t] = True
            outs.append(model.lm_head(model.hidden(cont[:, t:t + 1], None, None, mask=kv[:, None, None, :L + t + 1],
                                                   cache=cache, cache_len=L + t))[:, -1])
        got = torch.stack(outs, 1)
        return d1, max((got[i] - full[i]).abs().max().item() for i in range(2))


BACKENDS = {"dec": DecBackend, "encdec2": ED2Backend, "encdec": EDNewBackend, "txl": TXLBackend}


# ============================================================== generation: logits processing (shared by all backends)
class Decoder:
    """Per-step logits processing for a batch: banned specials, min length, repetition penalty, no-repeat n-gram,
    greedy / top-k / top-p sampling; stops a row at EOS."""

    def __init__(self, B, V, g, ids_cfg, device):
        self.B, self.V, self.g, self.dev = B, V, g, device
        self.eos = ids_cfg["eos"]
        self.ban = sorted({ids_cfg["pad"], ids_cfg["bos"], ids_cfg["unk"]} - {self.eos})
        self.seen = torch.zeros(B, V, dtype=torch.bool, device=device)
        self.hist = [[] for _ in range(B)]
        self.maps = [dict() for _ in range(B)]
        self.done = torch.zeros(B, dtype=torch.bool, device=device)
        self.gen = torch.Generator(device=device)
        self.gen.manual_seed(int(g.get("seed", 0)))

    def step(self, logits, t):
        g, lg = self.g, logits
        lg[:, self.ban] = float("-inf")
        if t < g.get("min_new", 0):
            lg[:, self.eos] = float("-inf")
        rp = g.get("rep_penalty", 1.0)
        if rp != 1.0:
            lg = torch.where(self.seen, torch.where(lg > 0, lg / rp, lg * rp), lg)
        n = g.get("no_repeat_ngram", 0)
        if n > 1 and t >= n - 1:
            rows, cols = [], []
            for b in range(self.B):
                h = self.hist[b]
                ban = self.maps[b].get(tuple(h[-(n - 1):]))
                if ban:
                    rows += [b] * len(ban)
                    cols += list(ban)
            if rows:
                lg[rows, cols] = float("-inf")
        T = g.get("temperature", 0.0)
        if T <= 0:
            nxt = lg.argmax(-1)
        else:
            lg = lg / T
            if g.get("top_k", 0) > 0:
                kth = torch.topk(lg, g["top_k"], -1).values[:, -1:]
                lg = lg.masked_fill(lg < kth, float("-inf"))
            if g.get("top_p", 1.0) < 1.0:
                sl, si = torch.sort(lg, descending=True, dim=-1)
                pr = sl.softmax(-1)
                sl = sl.masked_fill((pr.cumsum(-1) - pr) > g["top_p"], float("-inf"))
                lg = torch.full_like(lg, float("-inf")).scatter(-1, si, sl)
            nxt = torch.multinomial(lg.softmax(-1), 1, generator=self.gen).squeeze(1)
        nxt = torch.where(self.done, torch.full_like(nxt, self.eos), nxt)
        self.seen[torch.arange(self.B, device=self.dev), nxt] = True
        nl, dn = nxt.tolist(), self.done.tolist()
        for b in range(self.B):
            if dn[b]:
                continue
            h = self.hist[b]
            h.append(nl[b])
            if n > 1 and len(h) >= n:
                self.maps[b].setdefault(tuple(h[-n:-1]), set()).add(h[-1])
        self.done |= nxt == self.eos
        return nxt

    def all_done(self):
        return bool(self.done.all())

    def outputs(self):
        return [[x for x in h if x != self.eos] for h in self.hist]


def generate_texts(backend, model, docs, J, g, device, dec_fn, bs=48, log=None):
    prompts = [backend.gen_prompt(d, J) for d in docs]
    order = sorted(range(len(docs)), key=lambda i: len(prompts[i]))
    out = [None] * len(docs)
    t0 = time.time()
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        for i, ids in zip(idx, backend.generate(model, [prompts[i] for i in idx], g, device, J["_ids"])):
            out[i] = " ".join(dec_fn(ids).split())
    for i, d in enumerate(docs):                             # an empty output would score 0: fall back to the selector
        if not out[i]:
            out[i] = " ".join(context_text(d, 600, "learned").split()[:100])
    secs = time.time() - t0
    if log:
        log(f"generated {len(docs)} summaries in {secs:.0f}s ({1000 * secs / max(1, len(docs)):.0f} ms/doc) | {g}")
    return out, secs


# ============================================================== adaptation methods
def _slots(model, names):
    out = []
    for blk in model.blocks:
        out += P2._linear_slots(blk, names)
    return out


def apply_method(model, method, J):
    """Freeze / wrap for the method. Returns the new PEFT parameters (they get `peft_lr`)."""
    for p in model.parameters():
        p.requires_grad_(method in ("full", "kd"))
    new = []
    if method in ("lora", "lorma"):
        sl = _slots(model, J["lowrank_targets"])
        assert sl, "no LoRA/LoRMA target found"
        for holder, nm in sl:
            m = P2.LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method)
            setattr(holder, nm, m)
            new += [m.A, m.B]
    elif method == "qlora":
        sl = _slots(model, J["lowrank_targets"])
        for holder, nm in sl:
            m = P2.LowRankLinear(NF4Linear(getattr(holder, nm), J.get("nf4_block", 64)), J["rank"], J["alpha"],
                                 J["lowrank_dropout"], "lora")
            setattr(holder, nm, m)
            new += [m.A, m.B]
    elif method == "adapter":
        sl = _slots(model, J["adapter_targets"])
        assert sl, "no adapter target found"
        for holder, nm in sl:
            m = P2.Adapter(getattr(holder, nm), J["adapter_dim"])
            setattr(holder, nm, m)
            new += list(m.down.parameters()) + list(m.up.parameters())
    elif method in ("prompt", "prefix"):
        new = model.add_prompts(J["n_prompt"], deep=(method == "prefix"), seed=J["seed"])
    elif method == "adalora":
        from peft import AdaLoraConfig
        from peft.tuners.adalora import AdaLoraModel
        total = J["_total_steps"]
        cfg = AdaLoraConfig(target_modules=J["lowrank_targets"], init_r=J["ada_init_r"], target_r=J["ada_target_r"],
                            lora_alpha=J["alpha"], lora_dropout=J["lowrank_dropout"], total_step=total,
                            tinit=int(0.1 * total), tfinal=int(0.2 * total), deltaT=max(1, int(0.02 * total)),
                            orth_reg_weight=J["ada_orth_reg"])
        ada = AdaLoraModel(model, cfg, "default")
        object.__setattr__(model, "_ada", ada)
        for n_, p in model.named_parameters():
            if "lora_" in n_:
                p.requires_grad_(True)
                new.append(p)
        assert new, "AdaLoRA found no target modules"
    elif method in ("full", "kd", "frozen"):
        pass
    else:
        raise ValueError(method)
    return new


def adalora_orth_reg(model):
    reg, n = 0.0, 0
    for name, p in model.named_parameters():
        if ("lora_A" in name or "lora_B" in name) and p.requires_grad and p.dim() == 2:
            pc = p @ p.T if "lora_A" in name else p.T @ p
            reg = reg + torch.norm(pc - torch.eye(pc.size(0), device=p.device, dtype=pc.dtype), p="fro")
            n += 1
    return reg / max(n, 1)


def build_model(J, backend, method):
    layers = J["kd_student_layers"] if method == "kd" else None
    model = backend.build(J, layers)
    p1_ids = {id(p) for p in model.parameters()}
    new = apply_method(model, method, J)
    model._p1_ids = p1_ids
    return model, new


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================== training
@torch.no_grad()
def dev_nll(backend, model, samples, device, bs):
    model.eval()
    tot, n = 0.0, 0
    for s in range(0, len(samples), bs):
        bt = backend.make_batch(samples[s:s + bs], model.n_prompt, device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            loss, k = backend.loss(model, bt)
        tot += float(loss) * k
        n += k
    model.train()
    return tot / max(1, n)


def train_one(J, backend, model, new_params, train_s, dev_s, device, log, teacher=None):
    torch.manual_seed(J["seed"])
    rng = np.random.default_rng(J["seed"])
    model.to(device)
    new_ids = {id(p) for p in new_params}
    peft_p = [p for p in model.parameters() if id(p) in new_ids]
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in new_ids]
    groups = []
    if peft_p:
        groups.append(dict(params=peft_p, lr=J["peft_lr"], weight_decay=0.0 if J["method"] in ("prompt", "prefix") else J["weight_decay"]))
    if body:
        groups.append(dict(params=body, lr=J["lr"], weight_decay=J["weight_decay"]))
    opt = torch.optim.AdamW(groups)
    for gr in opt.param_groups:
        gr["base_lr"] = gr["lr"]
    opt_ids = {id(p) for gr in opt.param_groups for p in gr["params"]}
    frozen_ids = {i for i in model._p1_ids if i not in opt_ids}
    keep_keys = ({n for n, p in model.named_parameters(remove_duplicate=False) if id(p) not in frozen_ids}
                 | {n for n, _ in model.named_buffers()})
    model._keep_keys = keep_keys
    B = J["batch_size"]
    planned = max(10, int(J["epochs"] * math.ceil(len(train_s) / B)))
    total = planned
    warm = max(1, int(J["warmup"] * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)
    ada = getattr(model, "_ada", None)

    def set_ada(tot):
        if ada is not None:
            c = ada.peft_config["default"]
            c.total_step, c.tinit, c.tfinal, c.deltaT = tot, int(0.1 * tot), int(0.2 * tot), max(1, int(0.02 * tot))
    set_ada(total)
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    best = dict(nll=float("inf"), step=0, state=None)
    evals, hist = [], []
    step, n_nan, n_skip, n_oom, tok_seen, docs_seen = 0, 0, 0, 0, 0, 0
    budget = J["train_hours"] * 3600
    t0 = time.time()
    t_cal = None
    next_eval = [int(total * (i + 1) / J["n_evals"]) for i in range(J["n_evals"])]
    log(f"{len(train_s)} training documents, batch {B} -> {planned} steps for {J['epochs']} epochs (time budget {J['train_hours']:.2f} h)")

    def do_eval(tag):
        te = time.time()
        nll = dev_nll(backend, model, dev_s, device, J["eval_batch_size"])
        rec = dict(step=step, epoch=round(docs_seen / len(train_s), 3), dev_nll=round(nll, 4), secs=round(time.time() - te, 1), tag=tag)
        evals.append(rec)
        log("[EVAL] " + json.dumps(rec))
        if math.isfinite(nll) and nll < best["nll"]:
            best.update(nll=nll, step=step, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys})

    order = []
    model.train()
    tl, tn = 0.0, 0
    while step < total:
        if not order:
            order = rng.permutation(len(train_s)).tolist()
        idx = [order.pop() for _ in range(min(B, len(order)))]
        f = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))
        for gr in opt.param_groups:
            gr["lr"] = gr["base_lr"] * f
        while True:
            try:
                bt = backend.make_batch([train_s[i] for i in idx], model.n_prompt, device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    loss, k = backend.loss(model, bt, teacher, J)
                if ada is not None and J["ada_orth_reg"] > 0:
                    loss = loss + J["ada_orth_reg"] * adalora_orth_reg(model)
                if not torch.isfinite(loss):
                    n_nan += 1
                    if n_nan <= 5 or n_nan % 100 == 0:
                        log(f"non-finite loss at step {step} (#{n_nan}) - batch skipped")
                    opt.zero_grad(set_to_none=True)
                    break
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                for gr in opt.param_groups:
                    torch.nn.utils.clip_grad_norm_([p for p in gr["params"] if p.grad is not None], 1.0)
                sc0 = scaler.get_scale() if use_amp else 1.0
                scaler.step(opt)
                scaler.update()
                if use_amp and scaler.get_scale() < sc0:
                    n_skip += 1
                if ada is not None:
                    ada.update_and_allocate(step)
                tl += float(loss.detach()); tn += 1; tok_seen += k
                break
            except torch.cuda.OutOfMemoryError:
                n_oom += 1
                opt.zero_grad(set_to_none=True)
                loss = bt = None
                torch.cuda.empty_cache()
                if not model._ckpt:
                    model._ckpt = True
                    log(f"CUDA OOM at step {step} -> gradient checkpointing ON")
                elif B > 1:
                    B = max(1, B // 2)
                    idx = idx[:B]
                    log(f"CUDA OOM again -> batch size {B}")
                else:
                    raise
        step += 1
        docs_seen += len(idx)
        if step == 10:
            t_cal = time.time()
        if step == J["calib_steps"] and t_cal is not None:
            sps = (time.time() - t_cal) / (step - 10)
            eval_s = J["n_evals"] * sps * len(dev_s) / B / 2.5
            fit = step + int((budget - (time.time() - t0) - eval_s) / sps)
            total = max(step + 10, min(planned, fit))
            warm = min(warm, max(1, int(J["warmup"] * total)))
            next_eval = [max(step + 1, int(total * (i + 1) / J["n_evals"])) for i in range(J["n_evals"])]
            set_ada(total)
            log(f"calibration: {sps:.3f} s/step | steps {planned} planned -> {total} fit the {J['train_hours']:.2f} h budget "
                f"(= {total * B / len(train_s):.2f} epochs)")
        if step % J["log_every"] == 0:
            el = time.time() - t0
            log(json.dumps(dict(step=step, total=total, loss=round(tl / max(tn, 1), 4), lr_f=round(f, 4), tok_s=int(tok_seen / max(el, 1e-6)),
                                min=round(el / 60, 1), nan=n_nan, skip=n_skip, ckpt=model._ckpt)))
            hist.append(dict(step=step, loss=tl / max(tn, 1), min=el / 60))
            tl, tn = 0.0, 0
        if step in next_eval or step >= total:
            do_eval("scheduled")
            next_eval = [s for s in next_eval if s > step]
            if J.get("early_check") and len(evals) == 1 and not math.isfinite(evals[0]["dev_nll"]):
                log("!!! dev NLL is not finite after the first evaluation -> collapsed, stopping this job")
                break
        elif time.time() - t0 > 1.08 * budget:
            log(f"time budget exceeded at step {step}/{total} -> final eval and stop")
            do_eval("time-stop")
            break
    if best["state"] is None:
        best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys}
    info = dict(train_secs=time.time() - t0, steps=step, planned_steps=planned, docs_seen=docs_seen, epochs=docs_seen / len(train_s),
                tok_per_s=tok_seen / max(time.time() - t0, 1e-6), nan_batches=n_nan, skipped_steps=n_skip, ooms=n_oom,
                grad_ckpt=model._ckpt, batch_size=B, history=hist, evals=evals,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2 ** 30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def write_zip(pred, path):
    path = Path(path)
    pj = path.with_suffix(".json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    os.remove(pj)
    return path


def check_prediction(pred, test_docs):
    assert isinstance(pred, list) and len(pred) == len(test_docs), "one summary per test document"
    assert [p["id"] for p in pred] == [d["id"] for d in test_docs], "ids must be exactly the test ids (same order)"
    for p in pred:
        assert isinstance(p["summary"], str) and p["summary"].strip(), f"{p['id']}: empty summary"


def subsample(docs, n, seed):
    if n is None or len(docs) <= n:
        return docs
    idx = np.random.default_rng(seed).choice(len(docs), n, replace=False)
    return [docs[i] for i in sorted(idx)]


def analysis(dev, preds, res, device, cuts=(100, 200, 300, 400)):
    """Generation length vs metrics (outputs cut to N words), document length buckets, input coverage."""
    refs = [d["ref"] for d in dev]
    docs_text = ["".join(d["units"]) for d in dev]
    out = {}
    for c in cuts:
        cut = [" ".join(p.split()[:c]) for p in preds]
        r = score_outputs(cut, refs, docs_text, device)
        out[f"first_{c}_words"] = {k: round(r[k], 4) for k in ("bertscore_f1", "rouge1", "rougeL", "words") if k in r}
    pd_ = res["per_doc"]
    key = "bertscore_f1" if "bertscore_f1" in pd_[0] else "rouge1"
    nw = np.asarray([d["n_words"] for d in dev])
    by_len = {}
    for name, lo, hi in [("<2k words", 0, 2000), ("2k-5k", 2000, 5000), (">=5k", 5000, 10 ** 9)]:
        m = (nw >= lo) & (nw < hi)
        if m.any():
            by_len[name] = dict(docs=int(m.sum()), **{k: round(float(np.mean([pd_[i][k] for i in np.where(m)[0]])), 4)
                                                     for k in (key, "rouge1", "rougeL")})
    return dict(length_cut=out, by_doc_length=by_len, metric=key)


def run_baselines(J, D, dev, test, device, log, dec_fn, backend, out):
    """No training: (1) the learned selector's sentences as an EXTRACTIVE summary, (2) lead (head of the document),
    (3) ZERO-SHOT Phase-1 model (no fine-tuning) continuing '<context> Summary:'."""
    R = dict(name=J["name"], method="baselines")
    refs, dtext = [d["ref"] for d in dev], ["".join(d["units"]) for d in dev]
    nw = J["extract_words"]
    ext = lambda d, mode: " ".join(context_text(d, 4 * nw, mode).split()[:nw])
    for mode in ("learned", "head"):
        dp = [ext(d, mode) for d in dev]
        r = score_outputs(dp, refs, dtext, device)
        R[f"extractive_{mode}"] = {k: v for k, v in r.items() if k != "per_doc"}
        log(f"extractive ({mode}, {nw} words): " + json.dumps({k: round(v, 4) for k, v in R[f'extractive_{mode}'].items()}))
        tp = [dict(id=d["id"], summary=ext(d, mode)) for d in test]
        check_prediction(tp, test)
        write_zip(tp, out / f"baseline_extractive_{mode}.zip")
    model = backend.build(J).to(device)
    g = dict(J["decode_grid"][J["zeroshot_decode"]], max_new=J["sum_tokens"])
    dp, _ = generate_texts(backend, model, dev, J, g, device, dec_fn, J["gen_batch_size"], log)
    r = score_outputs(dp, refs, dtext, device)
    R["zeroshot"] = {k: v for k, v in r.items() if k != "per_doc"}
    log("zero-shot Phase-1 model: " + json.dumps({k: round(v, 4) for k, v in R["zeroshot"].items()}))
    json.dump(dict(zip([str(d["id"]) for d in dev], dp)), open(out / "dev_pred_zeroshot.json", "w"))
    R["collapsed"] = True                                    # never an upload candidate
    json.dump(R, open(out / "results.json", "w"), indent=1, default=str)
    log("DONE -> baselines (not for upload)")


def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    spec = D["spec"]
    tok = P2.load_bpe(spec["tok_py"], spec["tok_path"], norm=spec["norm"])
    _, enc_fn = text_fns(tok, spec["arch"])
    dec_fn = decode_fn(tok)
    J["_ids"] = special_ids(tok)
    J["_eos"] = J["_ids"]["eos"]
    backend = BACKENDS[J["arch"]]()
    J["_sep"] = backend.sep_ids(J, enc_fn)
    method = J["method"]
    P = J["n_prompt"] if method in ("prompt", "prefix") else 0
    J["_ctx_budget"] = backend.ctx_budget(J, len(J["_sep"]), P)
    train, dev, test = D["splits"]["train"], D["splits"]["dev"], D["splits"]["test"]
    if J.get("quick"):
        q = J["quick_docs"]
        train, dev, test = subsample(train, q, 1), subsample(dev, max(12, q // 4), 2), subsample(test, max(10, q // 6), 3)
    log(f"arch={J['arch']} method={method} device={device} | train {len(train)} dev {len(dev)} test {len(test)} | context "
        f"{J['_ctx_budget']} tokens ({J['ctx_select']}) + summary {J['sum_tokens']} tokens | budget {J['train_hours']:.2f} h")
    if method == "baselines":
        return run_baselines(J, D, dev, test, device, log, dec_fn, backend, out)
    R = dict(name=J["name"], method=method, config={k: v for k, v in J.items() if not k.startswith("_")})
    train_s = [backend.train_sample(d, J) for d in train]
    dev_s = [backend.train_sample(d, J) for d in subsample(dev, J["eval_dev_docs"], 4)]
    R["targets_complete_pct"] = round(100 * float(np.mean([len(d["sum_ids"]) <= J["sum_tokens"] for d in train])), 1)
    J["_total_steps"] = max(10, int(J["epochs"] * len(train) / J["batch_size"]))

    teacher = None
    if method == "kd":                                       # teacher = the best HEALTHY fine-tuned full-depth model
        cands = J["kd_teacher_dir"] if isinstance(J["kd_teacher_dir"], list) else [J["kd_teacher_dir"]]
        ok = []
        for c in map(Path, cands):
            if (c / "val_model.pt").exists() and (c / "results.json").exists():
                rc = json.load(open(c / "results.json"))
                key = rc.get("dev", {}).get("key", -1.0)
                log(f"teacher candidate {c.name}: dev score {key:.4f} (nll {rc.get('dev_nll')}), collapsed={rc.get('collapsed')}")
                if J.get("quick") or not rc.get("collapsed"):
                    ok.append((key, -rc.get("dev_nll", 99), str(c)))
        assert ok, f"KD needs a healthy teacher checkpoint in one of {cands}"
        tdir = Path(max(ok)[2])
        tj = json.load(open(tdir / "job.json"))
        tj["_total_steps"] = 1000
        teacher, _ = build_model(tj, backend, tj["method"])
        miss, unexp = teacher.load_state_dict(torch.load(tdir / "val_model.pt", map_location="cpu"), strict=False)
        assert not unexp, f"teacher checkpoint has unexpected keys {unexp[:5]}"
        assert teacher.n_prompt == 0
        teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        tres = json.load(open(tdir / "results.json"))
        R["teacher"] = dict(name=tj["name"], method=tj["method"], params=count_params(teacher)[0], dev=tres.get("dev", {}).get("key"),
                            ms_per_doc=tres.get("gen_ms_per_doc"))
        log(f"teacher {tj['name']} ({tj['method']}) {R['teacher']['params'] / 1e6:.2f}M params, dev score {R['teacher']['dev']}")

    model, new = build_model(J, backend, method)
    R["params_total"], R["params_trainable"] = count_params(model)
    if method == "qlora":
        nf = [m for m in model.modules() if isinstance(m, NF4Linear)]
        R["qlora"] = dict(quantised_layers=len(nf), weights=sum(m.n for m in nf), nf4_bytes=sum(m.nbytes() for m in nf),
                          fp16_bytes=2 * sum(m.n for m in nf))
        R["params_total"] += R["qlora"]["weights"]
        log(f"QLoRA: {len(nf)} projections in NF4: {R['qlora']['nf4_bytes'] / 2 ** 20:.1f} MB instead of {R['qlora']['fp16_bytes'] / 2 ** 20:.1f} MB fp16")
    log(f"params total={R['params_total'] / 1e6:.2f}M trainable={R['params_trainable'] / 1e6:.3f}M | PEFT tensors {len(new)} | "
        f"{R['targets_complete_pct']}% of the training summaries fit completely in {J['sum_tokens']} tokens")

    best, info = train_one(J, backend, model, new, train_s, dev_s, device, log, teacher)
    del teacher
    R["train"] = info
    model.load_state_dict(best["state"], strict=False)
    R["dev_nll"], R["best_step"] = best["nll"], best["step"]
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    model.to(device).eval()
    for m in model.modules():
        if isinstance(m, NF4Linear):
            m.cache_dense(torch.float16 if device.type == "cuda" else torch.float32)
    refs = [d["ref"] for d in dev]
    dtext = ["".join(d["units"]) for d in dev]

    # ---- decoding: choose on a dev subset, then the whole dev split, then test
    sw = subsample(list(range(len(dev))), J["sweep_docs"], 6)
    sweep = {}
    for name in J["decode_sweep"]:
        g = dict(J["decode_grid"][name], max_new=J["sum_tokens"])
        pr, secs = generate_texts(backend, model, [dev[i] for i in sw], J, g, device, dec_fn, J["gen_batch_size"], log)
        r = score_outputs(pr, [refs[i] for i in sw], [dtext[i] for i in sw], device)
        sweep[name] = {k: round(v, 4) for k, v in r.items() if k != "per_doc"}
        log(f"[decode] {name}: " + json.dumps(sweep[name]))
    R["decode_sweep"] = sweep
    pick = max(sweep, key=lambda n: sweep[n]["key"])
    g = dict(J["decode_grid"][pick], max_new=J["sum_tokens"])
    R["decode"] = dict(name=pick, **g)
    pdv, secs = generate_texts(backend, model, dev, J, g, device, dec_fn, J["gen_batch_size"], log)
    res = score_outputs(pdv, refs, dtext, device)
    R["dev"] = {k: v for k, v in res.items() if k != "per_doc"}
    R["analysis"] = analysis(dev, pdv, res, device)
    ctx_cov = [content_recall(context_text(d, J["_ctx_budget"], J["ctx_select"]), d["ref_prefix_words"]) for d in dev]
    R["analysis"]["input_coverage"] = dict(mean=round(float(np.mean(ctx_cov)), 4),
                                           corr_with_output_rouge1_recall=round(float(np.corrcoef(ctx_cov, [m["rouge1_recall"] for m in res["per_doc"]])[0, 1]), 3)
                                           if len(dev) > 2 else None)
    json.dump(dict(zip([str(d["id"]) for d in dev], pdv)), open(out / "dev_pred.json", "w"))
    t0 = time.time()
    pts, _ = generate_texts(backend, model, test, J, g, device, dec_fn, J["gen_batch_size"], log)
    R["gen_ms_per_doc"] = 1000 * (time.time() - t0) / max(1, len(test))
    R["gen_peak_mem_gb"] = (torch.cuda.max_memory_allocated(device) / 2 ** 30) if device.type == "cuda" else None
    pred = [dict(id=d["id"], summary=s) for d, s in zip(test, pts)]
    check_prediction(pred, test)
    write_zip(pred, out / f"submission_{J['name']}.zip")
    json.dump(dict(zip([str(d["id"]) for d in test], pts)), open(out / "test_pred.json", "w"))
    tw = [len(s.split()) for s in pts]
    R["test_pred"] = dict(words_mean=round(float(np.mean(tw)), 1), words_min=int(min(tw)), rep4=round(float(np.mean([rep_ngram(words(s)) for s in pts])), 4))
    R["collapsed"] = bool(not math.isfinite(R["dev_nll"]) or R["dev"]["words"] < 15 or R["dev"]["rep4"] > 0.5
                          or R["dev"].get("bertscore_f1", 1.0) < 0.70 or R["test_pred"]["words_mean"] < 15)
    json.dump(R, open(out / "results.json", "w"), indent=1, default=str)
    d_ = R["dev"]
    log(f"DEV BERTScore-F1 {d_.get('bertscore_f1', float('nan')):.4f} | ROUGE-1 {d_['rouge1']:.4f} ROUGE-2 {d_['rouge2']:.4f} ROUGE-L {d_['rougeL']:.4f} "
        f"| {d_['words']:.0f} words | rep4 {d_['rep4']:.3f} | novel bigrams {d_['novel_bigrams']:.3f} | numbers in source {d_['number_precision']:.3f} "
        f"| decode '{pick}' | nll {R['dev_nll']:.4f}")
    log(f"DONE -> submission_{J['name']}.zip | test: {R['test_pred']}" + (" | !!! COLLAPSED - do NOT upload" if R["collapsed"] else ""))


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p6/p6_summ.py


In [4]:
# ============================================================ 2. DATA + TOKENISATION CHECKS + SENTENCE UNITS + CONTEXT SELECTOR + MODEL CHECK
sys.path.insert(0, str(WORK))
import importlib, p2_ner as P2, p6_summ as P6; importlib.reload(P2); importlib.reload(P6)
import torch, numpy as np, pandas as pd
SUM_TOKENS, MAX_LEN, SEP_TEXT, N_DEV = 480, 1024, " Summary:", 400

# (a) data: drop train rows that are test documents / duplicates / empty, hold out N_DEV train documents as dev
data = P6.load_summ(DATA, n_dev=N_DEV)
if QUICK_TEST:
    data["train"], data["dev"] = P6.subsample(data["train"], 400, 0), P6.subsample(data["dev"], 60, 0)
print("cleaning:", data["stats"]["cleaning"], "| train", len(data["train"]), "dev", len(data["dev"]), "test", len(data["test"]))
assert not ({str(d["id"]) for d in data["train"] + data["dev"]} & {str(d["id"]) for d in data["test"]})

# (b) token ids must be identical to the Phase-1 pipeline (documents AND summaries)
tok = P2.load_bpe(TOK_PY, TOK_PATH, norm=NORM)
print("tokenizer kind:", tok.kind, "| normalisation:", NORM, "| vocab", tok.vocab_size); assert tok.vocab_size == 32000
NORM_FN, ENC_FN = P6.text_fns(tok, ARCH)
IDS = P6.special_ids(tok); print("special ids:", IDS)
sample = [d["document"][:8000] for d in data["dev"][:4]] + [d["summary"][:4000] for d in data["dev"][:4]]
bad = [s for s in sample if list(ENC_FN(NORM_FN(s))) != list(tok.reference_encode(s))]
print(f"token-id check on {len(sample)} texts: {len(sample) - len(bad)} identical to the Phase-1 encoding"); assert not bad

# (c) sentence units: exact substrings, and unit-by-unit encoding == encoding of the whole document
for d in data["dev"][:8]:
    t = NORM_FN(d["document"]); units = P6.split_units(t)
    assert "".join(units) == t, "units do not rebuild the text"
    assert [i for u in units for i in ENC_FN(u)] == list(ENC_FN(t)), "unit-wise encoding differs from the full encoding"
print("sentence units rebuild the text exactly and give identical token ids")

# (d) tokenise + split + selector features (all CPU cores)
SPEC = dict(arch=ARCH, tok_py=TOK_PY, tok_path=TOK_PATH, norm=NORM, label_prefix_words=400, sum_prefix=" ")
t0 = time.time()
splits = P6.prepare(data, SPEC)
print(f"prepared in {time.time() - t0:.0f}s")
def _st(docs, key):
    a = np.asarray([key(d) for d in docs]); return f"p10 {int(np.percentile(a, 10))} | p50 {int(np.median(a))} | p90 {int(np.percentile(a, 90))} | max {int(a.max())}"
print("document tokens:", _st(splits["train"], lambda d: d["n_tok"]), "| tokens per word", round(sum(d["n_tok"] for d in splits["train"]) / max(1, sum(d["n_words"] for d in splits["train"])), 3))
print("summary tokens :", _st(splits["train"], lambda d: len(d["sum_ids"])), f"| complete within {SUM_TOKENS} tokens: "
      f"{100 * np.mean([len(d['sum_ids']) <= SUM_TOKENS for d in splits['train']]):.1f}%")
print("sentence units per document:", _st(splits["train"], lambda d: len(d["units"])))

# (e) context selector + how much of the reference start each long-document strategy shows the model
CTX_BUDGET = P6.BACKENDS[ARCH]().ctx_budget(dict(max_len=MAX_LEN, sum_tokens=SUM_TOKENS), len(ENC_FN(SEP_TEXT)) if SEP_TEXT else 0, 0)
sel = P6.train_selector(splits["train"])
for sp in splits.values(): P6.score_units(sp, sel)
cov = P6.coverage_table(splits["dev"], CTX_BUDGET)
json.dump(dict(budget_tokens=CTX_BUDGET, table=cov), open(WORK / "context_coverage.json", "w"), indent=1)
print(f"\nCONTEXT COVERAGE (share of the reference's first 400 words' content words that appear in the {CTX_BUDGET}-token context):")
display(pd.DataFrame(cov).T)
CTX_SELECT = max([m for m in cov if m in ("head", "tail", "headtail", "central", "learned")], key=lambda m: cov[m]["content_recall"])
print("-> context selection used by the jobs:", CTX_SELECT)
for sp in splits.values():
    for d in sp: d.pop("feats", None)
pickle.dump(dict(spec=SPEC, splits=splits), open(CACHE, "wb"), protocol=4)
print("cache:", CACHE, f"{CACHE.stat().st_size / 1e6:.0f} MB")

# (f) the summariser must reproduce the Phase-1 forward pass, and KV-cache generation must equal the full forward pass
J0 = dict(arch=ARCH, model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT, dropout=0.0)
base = P2.load_backbone(J0).eval()
print(f"Phase-1 model: {sum(p.numel() for p in base.parameters()) / 1e6:.2f}M params")
mdl = P6.BACKENDS[ARCH]().build(J0).eval()
d1, d2 = P6.BACKENDS[ARCH]().check_equivalence(P2.load_backbone(J0).eval(), mdl, torch.device("cpu"))
print(f"summariser vs Phase-1 forward max |diff| = {d1:.2e} | KV-cache vs full forward max |diff| = {d2:.2e}"); assert d1 < 1e-4 and d2 < 1e-3
del base, mdl


{"train_rows": 7030, "kept": 7001, "train": 6601, "dev": 400, "test": 100, "cleaning": {"dropped_duplicate_document": 27, "dropped_empty_document_or_summary": 2}}
cleaning: {'dropped_duplicate_document': 27, 'dropped_empty_document_or_summary': 2} | train 6601 dev 400 test 100
tokenizer kind: old | normalisation: ascii | vocab 32000
special ids: {'pad': 0, 'unk': 1, 'bos': 2, 'eos': 3}
token-id check on 8 texts: 8 identical to the Phase-1 encoding
sentence units rebuild the text exactly and give identical token ids
[prepare] train: 6601 documents in 63s
[prepare] dev: 400 documents in 5s
[prepare] test: 100 documents in 2s
prepared in 75s
document tokens: p10 1580 | p50 3668 | p90 9137 | max 162067 | tokens per word 1.166
summary tokens : p10 343 | p50 755 | p90 1795 | max 32478 | complete within 480 tokens: 23.0%
sentence units per document: p10 49 | p50 109 | p90 273 | max 4384
[selector] trained on 400000 sentence units in 11s

CONTEXT COVERAGE (share of the reference's first 400 wo

,content_recall
head,0.4397
tail,0.4038
headtail,0.4211
central,0.4410
learned,0.4727
oracle,0.7205
whole document,0.9013


-> context selection used by the jobs: learned
cache: /tmp/p6_cache/cache.pkl 373 MB
Phase-1 model: 64.21M params
summariser vs Phase-1 forward max |diff| = 0.00e+00 | KV-cache vs full forward max |diff| = 2.15e-05


In [5]:
# ============================================================ 3. EXPERIMENTS (2 GPUs -> 2 queues)
DECODE_GRID = {
    "greedy_ng4":        dict(temperature=0.0, no_repeat_ngram=4),
    "greedy_ng3_rp1.1":  dict(temperature=0.0, no_repeat_ngram=3, rep_penalty=1.1),
    "greedy_ng4_rp1.2":  dict(temperature=0.0, no_repeat_ngram=4, rep_penalty=1.2),
    "greedy_ng4_min300": dict(temperature=0.0, no_repeat_ngram=4, min_new=300),
    "t0.7_p0.9_ng4":     dict(temperature=0.7, top_p=0.9, no_repeat_ngram=4, seed=0),
}
COMMON = dict(
    arch=ARCH, cache=str(CACHE), model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT,
    sum_tokens=SUM_TOKENS, sep_text=SEP_TEXT, ctx_select=CTX_SELECT,
    max_len=1024,                                   # one sequence: [soft prompt] + context + " Summary:" + summary
    txl_clamp=511, grad_ckpt=True,                  # relative distances clamped at 511 (largest seen in Phase 1); L^2 attention -> checkpointing
    lowrank_targets=["q_net", "kv_net", "o_net", "ff1", "ff2"],   # LoRA / QLoRA / AdaLoRA
    adapter_targets=["o_net", "ff2"],               # Houlsby: after the attention output and the FFN output
    kd_student_layers=[0, 3, 6, 9, 11, 13],         # KD student: 6 of the 14 Phase-1 layers
    dropout=0.1, batch_size=8, eval_batch_size=8, epochs=5, warmup=0.05, weight_decay=0.01,
    lr=3e-5, peft_lr=3e-4, rank=16, alpha=32, lowrank_dropout=0.05, adapter_dim=64, n_prompt=20,
    ada_init_r=12, ada_target_r=8, ada_orth_reg=0.1, nf4_block=64, kd_temperature=2.0, kd_alpha=0.5,
    calib_steps=40, n_evals=5, eval_dev_docs=400, log_every=100, seed=781, early_check=True,
    decode_grid=DECODE_GRID, decode_sweep=list(DECODE_GRID), sweep_docs=120, gen_batch_size=48,
    zeroshot_decode="greedy_ng4", extract_words=400, gen_reserve_h=0.3,
    train_hours=None, save_model=False, quick=QUICK_TEST, quick_docs=64,
)
# LoRMA multiplies a projection's OUTPUT: y = (I + s*B*A) W x. On residual-stream projections this overflowed fp16 in
# Phase 4, so LoRMA adapts the attention-side projections with s = alpha/r = 1 (16/16).
T_DIRS = [str(OUT / "lorma"), str(OUT / "full")]
ALT_CTX = "head" if CTX_SELECT != "head" else "learned"   # the ablation uses the strategy the other jobs do NOT use
#               name            method       overrides
JOBS = {
  "lorma":      ("lorma",     dict(lowrank_targets=["q_net", "kv_net"], alpha=16, save_model=True)),        # compulsory + teacher
  "full":       ("full",      dict(lr=3e-5, save_model=True)),                                     # reference + teacher
  "kd":         ("kd",        dict(kd_teacher_dir=T_DIRS, lr=1e-4)),                                # compulsory
  "kd_small":   ("kd",        dict(kd_teacher_dir=T_DIRS, lr=1e-4, kd_student_layers=[0, 4, 9, 13])),
  "lora":       ("lora",      dict()),
  "qlora":      ("qlora",     dict()),
  "adalora":    ("adalora",   dict(peft_lr=1e-3)),
  "adapter_h":  ("adapter",   dict()),                                                             # Houlsby
  "adapter_p":  ("adapter",   dict(adapter_targets=["ff2"])),                                 # Pfeiffer
  "prompt":     ("prompt",    dict(peft_lr=3e-3, n_prompt=20)),
  "prefix":     ("prefix",    dict(peft_lr=1e-3, n_prompt=16)),
  "lorma_alt":  ("lorma",     dict(lowrank_targets=["q_net", "kv_net"], alpha=16, ctx_select=ALT_CTX)),    # long-document ablation
  "baselines":  ("baselines", dict(hours_scale=0.1)),                                              # analysis only
}
GROUPS = {   # queues (one per GPU; the KD teachers lorma + full run first) , hours per job
  "ALL": ([["lorma", "kd", "lora", "adapter_h", "prompt", "lorma_alt", "baselines"],
           ["full", "kd_small", "qlora", "adalora", "adapter_p", "prefix"]], 1.55),
  "A":   ([["lorma", "kd", "lora"], ["full", "kd_small", "adapter_h"]], 3.1),                # account 1
  "B":   ([["qlora", "adalora", "lorma_alt"], ["adapter_p", "prompt", "prefix", "baselines"]], 3.1),    # account 2
}
QUEUES, JOB_HOURS = GROUPS[GROUP]
RUN_ONLY = None           # e.g. ["prompt", "prefix"] to (re)run only those jobs
if RUN_ONLY:
    QUEUES = [[n for n in q if n in RUN_ONLY] for q in QUEUES]
print("group", GROUP, "| queues:", QUEUES, "| hours per job:", JOB_HOURS)

def teachers_of(n):
    t = JOBS[n][1].get("kd_teacher_dir")
    return [Path(x).name for x in (t if isinstance(t, list) else [t] if t else [])]

n_gpu = torch.cuda.device_count(); print("GPUs:", n_gpu)
if n_gpu < 2:                              # one queue: every teacher before every KD job
    QUEUES = [[n for q in QUEUES for n in q if not teachers_of(n)] + [n for q in QUEUES for n in q if teachers_of(n)]]
SCHEDULED, DONE = set(sum(QUEUES, [])), set()

def write_job(name, hours):
    meth, over = JOBS[name]
    over = dict(over); scale = over.pop("hours_scale", 1.0)
    J = dict(COMMON, name=name, method=meth, out_dir=str(OUT / name), **over)
    J["train_hours"] = max(0.05, hours * scale - J["gen_reserve_h"])
    if QUICK_TEST: J.update(epochs=1, n_evals=2, calib_steps=10 ** 9, log_every=5, train_hours=0.05, eval_dev_docs=12,
                            sweep_docs=8, decode_sweep=list(DECODE_GRID)[:2])
    Path(J["out_dir"]).mkdir(parents=True, exist_ok=True)
    p = Path(J["out_dir"]) / "job.json"; json.dump(J, open(p, "w"), indent=1)
    return p

def run_queue(names, gpu):
    for n in names:
        if (OUT / n / "results.json").exists():
            print(f"[skip] {n} already done"); DONE.add(n); continue
        wait = [t for t in teachers_of(n) if t in SCHEDULED]             # KD waits until its teacher jobs are finished
        if wait and not all(t in DONE for t in wait):
            print(f"[gpu{gpu}] {n} waits for its teacher job(s) {wait}")
            while not all(t in DONE for t in wait): time.sleep(30)
        try:
            run_one(n, gpu)
        finally:
            DONE.add(n)

def run_one(n, gpu):
    left = SESSION_H - (time.time() - T_START) / 3600 - 0.5          # keep 30 min for the summary + saving the output
    scale = JOBS[n][1].get("hours_scale", 1.0)
    hours = min(JOB_HOURS, (left - 0.1) / scale)
    if not QUICK_TEST and hours * scale < min(0.5, JOB_HOURS * scale):
        print(f"[gpu{gpu}] !!! not enough session time left for {n} ({left:.2f} h) - skipped; rerun it with RUN_ONLY"); return
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu) if n_gpu else "", PYTHONUNBUFFERED="1")
    p = subprocess.Popen([sys.executable, str(WORK / "p6_summ.py"), str(write_job(n, hours))], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" not in line and "warn(" not in line: print(f"[gpu{gpu}] {line}", end="")
    p.wait()
    if p.returncode != 0:
        print(f"[gpu{gpu}] !!! job {n} FAILED (see {OUT / n / 'log.txt'} and the lines above)")

T0 = time.time()
threads = [threading.Thread(target=run_queue, args=(q, g)) for g, q in enumerate(QUEUES)]
for t in threads: t.start()
for t in threads: t.join()
print(f"all jobs finished in {(time.time() - T0) / 60:.1f} min")


group ALL | queues: [['lorma', 'kd', 'lora', 'adapter_h', 'prompt', 'lorma_alt', 'baselines'], ['full', 'kd_small', 'qlora', 'adalora', 'adapter_p', 'prefix']] | hours per job: 1.55
GPUs: 2
[gpu1] [14:07:32][full] arch=txl method=full device=cuda | train 6601 dev 400 test 100 | context 541 tokens (learned) + summary 480 tokens | budget 1.25 h
[gpu0] [14:07:32][lorma] arch=txl method=lorma device=cuda | train 6601 dev 400 test 100 | context 541 tokens (learned) + summary 480 tokens | budget 1.25 h
[gpu0] [14:07:35][lorma] params total=64.89M trainable=0.688M | PEFT tensors 56 | 23.0% of the training summaries fit completely in 480 tokens
[gpu1] [14:07:35][full] params total=64.21M trainable=64.206M | PEFT tensors 0 | 23.0% of the training summaries fit completely in 480 tokens
[gpu1] [14:07:36][full] 6601 training documents, batch 8 -> 4130 steps for 5 epochs (time budget 1.25 h)
[gpu0] [14:07:36][lorma] 6601 training documents, batch 8 -> 4130 steps for 5 epochs (time budget 1.25 h)
[g

In [6]:
# ============================================================ 4. SUMMARY + SUBMISSIONS (+ consensus of the runs) + REPORT TABLES
import zipfile
SUB = Path("/kaggle/working/submissions" + ("_quick" if QUICK_TEST else "")); SUB.mkdir(exist_ok=True)
test_docs, dev_docs = splits["test"], splits["dev"]
if QUICK_TEST:
    test_docs = P6.subsample(test_docs, max(10, COMMON["quick_docs"] // 6), 3); dev_docs = P6.subsample(dev_docs, max(12, COMMON["quick_docs"] // 4), 2)
rows, sweeps, res = [], {}, {}
for n in JOBS:
    rp = OUT / n / "results.json"
    if not rp.exists() or n == "baselines": continue
    r = json.load(open(rp)); tr = r["train"]; res[n] = r
    z = OUT / n / f"submission_{n}.zip"
    with zipfile.ZipFile(z) as zz:
        assert zz.namelist() == ["prediction.json"]
        P6.check_prediction(json.loads(zz.read("prediction.json")), test_docs)
    shutil.copy(z, SUB / z.name)
    d = r["dev"]
    rows.append(dict(exp=n, method=r["method"], dev_BERTScore=round(d.get("bertscore_f1", float("nan")), 4), dev_ROUGE1=round(d["rouge1"], 4),
                     dev_ROUGE2=round(d["rouge2"], 4), dev_ROUGEL=round(d["rougeL"], 4), dev_nll=round(r["dev_nll"], 4), words=round(d["words"], 1),
                     rep4=round(d["rep4"], 4), novel_bigrams=round(d["novel_bigrams"], 3), numbers_in_source=round(d["number_precision"], 3),
                     decode=r["decode"]["name"], ctx=r["config"]["ctx_select"], epochs=round(tr["epochs"], 2), train_h=round(tr["train_secs"] / 3600, 2),
                     params_M=round(r["params_total"] / 1e6, 2), trainable_M=round(r["params_trainable"] / 1e6, 3), peak_GB=round(tr["peak_mem_gb"] or 0, 2),
                     gen_ms_per_doc=round(r["gen_ms_per_doc"], 1), test_words=r["test_pred"]["words_mean"],
                     UPLOAD="NO - collapsed" if r["collapsed"] else "yes"))
    sweeps[n] = {k: v.get("bertscore_f1", v["key"]) for k, v in r["decode_sweep"].items()}
df = pd.DataFrame(rows).sort_values("dev_BERTScore" if rows and not np.isnan(rows[0]["dev_BERTScore"]) else "dev_ROUGEL", ascending=False) if rows else pd.DataFrame()
pd.set_option("display.width", 320); pd.set_option("display.max_columns", 40)
display(df)
df.to_csv(WORK / ("summary_quick.csv" if QUICK_TEST else "summary.csv"), index=False)
if sweeps:
    print("DECODING (dev subset; BERTScore, or ROUGE if BERTScore is unavailable):"); display(pd.DataFrame(sweeps).T)
if (OUT / "baselines" / "results.json").exists():
    b = json.load(open(OUT / "baselines" / "results.json"))
    print("BASELINES (no training):")
    display(pd.DataFrame({k: {m: round(v[m], 4) for m in ("bertscore_f1", "rouge1", "rouge2", "rougeL", "words") if m in v}
                          for k, v in b.items() if isinstance(v, dict) and "rouge1" in v}).T)
print("CONTEXT COVERAGE of the long-document strategies:", json.load(open(WORK / "context_coverage.json"))["table"])
ok = [n for n in df["exp"] if not res[n]["collapsed"]] if len(df) else []
if ok:
    an = res[ok[0]]["analysis"]
    print(f"\nBEST RUN {ok[0]}: generation length vs score (outputs cut to N words):"); display(pd.DataFrame(an["length_cut"]).T)
    print("by document length:", an["by_doc_length"]); print("input coverage:", an["input_coverage"])

# extra submission: CONSENSUS of the healthy runs (for every document keep the summary that agrees most with the other runs'
# summaries, by ROUGE-1/2 overlap; an ensemble, NOT a new technique; no pretrained model involved). Kept only if it beats
# the best single run on dev.
CONS = [n for n in ok if (OUT / n / "dev_pred.json").exists()]
if len(CONS) >= 3:
    def consensus(preds):
        out = []
        for cs in zip(*preds):
            w = [P6.words(c) for c in cs]
            u = [sum((P6.rouge_n(w[i], w[j], 1)[2] + P6.rouge_n(w[i], w[j], 2)[2]) / 2 for j in range(len(cs)) if j != i) for i in range(len(cs))]
            out.append(cs[int(np.argmax(u))])
        return out
    ids_dev = [str(d["id"]) for d in dev_docs]
    dp = [[json.load(open(OUT / n / "dev_pred.json"))[i] for i in ids_dev] for n in CONS]
    cons_dev = consensus(dp)
    refs = [d["ref"] for d in dev_docs]; dtext = ["".join(d["units"]) for d in dev_docs]
    dev_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rc = P6.score_outputs(cons_dev, refs, dtext, dev_device)
    best_single = res[CONS[0]]["dev"]["key"]
    print(f"\nconsensus of {CONS}: dev score {rc['key']:.4f} vs best single run {CONS[0]} {best_single:.4f}")
    if rc["key"] > best_single:
        ids_test = [str(d["id"]) for d in test_docs]
        tp = [[json.load(open(OUT / n / "test_pred.json"))[i] for i in ids_test] for n in CONS]
        pred = [dict(id=d["id"], summary=s) for d, s in zip(test_docs, consensus(tp))]
        P6.check_prediction(pred, test_docs)
        P6.write_zip(pred, SUB / "submission_consensus.zip"); print("-> wrote submission_consensus.zip (upload it too)")
print("submission zips:", sorted(p.name for p in SUB.glob("*.zip")), "->", SUB)
if QUICK_TEST: print("\nQUICK TEST ONLY - do NOT submit these (they cover only a few test documents). Set QUICK_TEST=False and Save & Run All (Commit).")


,exp,method,dev_BERTScore,dev_ROUGE1,dev_ROUGE2,dev_ROUGEL,dev_nll,words,rep4,novel_bigrams,numbers_in_source,decode,ctx,epochs,train_h,params_M,trainable_M,peak_GB,gen_ms_per_doc,test_words,UPLOAD
1,full,full,0.8477,0.4543,0.2183,0.2338,2.2107,366.7,0.0129,0.334,0.642,greedy_ng3_rp1.1,learned,3.98,1.36,64.21,64.206,2.43,355.7,365.7,yes
5,qlora,qlora,0.8468,0.4439,0.2082,0.2259,2.2591,351.4,0.0237,0.339,0.647,greedy_ng4_rp1.2,learned,3.56,1.29,66.16,1.950,1.78,505.9,342.3,yes
4,lora,lora,0.8464,0.4355,0.2013,0.2208,2.2431,343.9,0.0188,0.355,0.660,greedy_ng4_rp1.2,learned,3.68,1.27,66.16,1.950,1.93,459.3,340.5,yes
2,kd,kd,0.8463,0.4267,0.2013,0.2219,2.5534,316.3,0.0237,0.330,0.682,greedy_ng4_rp1.2,learned,5.00,1.20,36.90,36.898,4.70,165.1,322.6,yes
6,adalora,adalora,0.8457,0.4302,0.1998,0.2195,2.2553,337.7,0.0213,0.349,0.645,greedy_ng4_rp1.2,learned,3.08,1.30,65.67,1.463,1.99,552.6,322.9,yes
7,adapter_h,adapter,0.8457,0.4507,0.2108,0.2278,2.2742,368.7,0.0113,0.364,0.579,greedy_ng3_rp1.1,learned,4.17,1.27,66.06,1.851,1.93,381.3,355.9,yes
0,lorma,lorma,0.8451,0.4507,0.2111,0.2262,2.3187,374.4,0.0260,0.338,0.620,greedy_ng4_rp1.2,learned,4.14,1.31,64.89,0.688,1.91,397.9,372.2,yes
8,adapter_p,adapter,0.8434,0.4276,0.1922,0.2147,2.2905,349.7,0.0198,0.398,0.585,greedy_ng4_rp1.2,learned,4.21,1.33,65.13,0.926,1.92,370.4,356.5,yes
10,prefix,prefix,0.8381,0.4329,0.1837,0.2086,2.3809,384.0,0.0085,0.470,0.535,greedy_ng3_rp1.1,learned,4.02,1.29,64.33,0.123,1.91,347.7,372.5,yes
11,lorma_alt,lorma,0.8380,0.4188,0.1737,0.2138,2.3523,380.1,0.0123,0.400,0.616,greedy_ng3_rp1.1,head,4.06,1.26,64.89,0.688,1.91,382.2,373.5,yes


DECODING (dev subset; BERTScore, or ROUGE if BERTScore is unavailable):


,greedy_ng4,greedy_ng3_rp1.1,greedy_ng4_rp1.2,greedy_ng4_min300,t0.7_p0.9_ng4
lorma,0.8438,0.8461,0.8464,0.8438,0.8451
full,0.8445,0.8486,0.8479,0.8445,0.8473
kd,0.8433,0.8469,0.8477,0.8428,0.8452
kd_small,0.8333,0.8383,0.8371,0.8329,0.8374
lora,0.8438,0.8469,0.8482,0.8436,0.8459
qlora,0.8436,0.8467,0.8479,0.8437,0.8470
adalora,0.8432,0.8462,0.8472,0.8431,0.8450
adapter_h,0.8444,0.8463,0.8461,0.8444,0.8448
adapter_p,0.8397,0.8436,0.8445,0.8396,0.8427
prompt,0.8318,0.8349,0.8344,0.8315,0.8340


BASELINES (no training):


,bertscore_f1,rouge1,rouge2,rougeL,words
extractive_learned,0.8501,0.4682,0.2402,0.2478,399.2425
extractive_head,0.8316,0.4270,0.1978,0.2121,399.2425
zeroshot,0.8224,0.3279,0.1262,0.1815,314.5150


CONTEXT COVERAGE of the long-document strategies: {'head': {'content_recall': 0.4397}, 'tail': {'content_recall': 0.4038}, 'headtail': {'content_recall': 0.4211}, 'central': {'content_recall': 0.441}, 'learned': {'content_recall': 0.4727}, 'oracle': {'content_recall': 0.7205}, 'whole document': {'content_recall': 0.9013}}

BEST RUN full: generation length vs score (outputs cut to N words):


,bertscore_f1,rouge1,rougeL,words
first_100_words,0.8414,0.2194,0.1450,99.9750
first_200_words,0.8489,0.3501,0.2025,199.4625
first_300_words,0.8495,0.4223,0.2267,292.5350
first_400_words,0.8479,0.4529,0.2337,361.4225


by document length: {'<2k words': {'docs': 100, 'bertscore_f1': 0.8548, 'rouge1': 0.5286, 'rougeL': 0.2738}, '2k-5k': {'docs': 204, 'bertscore_f1': 0.8476, 'rouge1': 0.4589, 'rougeL': 0.2345}, '>=5k': {'docs': 96, 'bertscore_f1': 0.8407, 'rouge1': 0.3672, 'rougeL': 0.1905}}
input coverage: {'mean': 0.4727, 'corr_with_output_rouge1_recall': 0.35}


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



consensus of ['full', 'qlora', 'lora', 'kd', 'adalora', 'adapter_h', 'lorma', 'adapter_p', 'prefix', 'lorma_alt', 'kd_small', 'prompt']: dev score 0.8491 vs best single run full 0.8477
-> wrote submission_consensus.zip (upload it too)
submission zips: ['submission_adalora.zip', 'submission_adapter_h.zip', 'submission_adapter_p.zip', 'submission_consensus.zip', 'submission_full.zip', 'submission_kd.zip', 'submission_kd_small.zip', 'submission_lora.zip', 'submission_lorma.zip', 'submission_lorma_alt.zip', 'submission_prefix.zip', 'submission_prompt.zip', 'submission_qlora.zip'] -> /kaggle/working/submissions
